# Notebook 03 - Justifikasi Pemilihan Hyperplane SVM

**Revisi Pengujian V3 - Skripsi Prediksi Diabetes**

Notebook ini menjawab pertanyaan penguji: **"Kenapa hyperplane pada SVM yang dipilih?"**

---

## Landasan teori singkat

Support Vector Machine (SVM) mencari sebuah **hyperplane pemisah**

$$ f(x) = w \cdot x + b = 0 $$

yang memisahkan kelas positif (diabetes) dan negatif (sehat) dengan **margin semaksimal mungkin**.
Lebar margin didefinisikan sebagai

$$ \text{margin} = \frac{2}{\lVert w \rVert} $$

sehingga memaksimalkan margin setara dengan **meminimalkan** $\tfrac{1}{2}\lVert w \rVert^2$.
Karena data medis nyata tidak pernah terpisah sempurna, dipakai formulasi **soft margin**:

$$ \min_{w,b,\xi} \; \tfrac{1}{2}\lVert w \rVert^2 + C \sum_i \xi_i $$

dengan $\xi_i$ = besar pelanggaran margin oleh sampel ke-$i$. Di sini:

- **C kecil** -> penalti pelanggaran ringan -> $\lVert w \rVert$ kecil -> **margin lebar**, model lebih halus/general, toleran terhadap noise.
- **C besar** -> penalti pelanggaran berat -> $\lVert w \rVert$ besar -> **margin sempit**, model mengikuti data latih (risiko overfitting).

Sedangkan **kernel** menentukan **ruang tempat hyperplane itu dicari**: kernel linear mencari
hyperplane di ruang fitur asli, sedangkan RBF/polinomial/sigmoid memetakan data ke ruang
berdimensi lebih tinggi lalu mencari hyperplane di sana (data yang tidak terpisah linear di
ruang asli bisa terpisah linear di ruang baru).

Dengan demikian, pertanyaan **"kenapa hyperplane ini yang dipilih"** secara teknis sama dengan
menjawab tiga hal secara empiris:

1. **Kernel mana** (ruang pencarian hyperplane) yang dipakai, dan apa buktinya kernel lain tidak lebih baik.
2. **Nilai C (dan gamma)** mana yang dipakai, dan apa konsekuensinya terhadap **lebar margin** serta **jumlah support vector**.
3. **Mengapa titik operasi itu** yang paling tepat untuk konteks skrining medis (sensitivitas/recall tinggi).

---

## Batasan komputasi (dilaporkan terbuka)

`SVC` dengan kernel non-linear memiliki kompleksitas pelatihan sekitar **O(n^2) sampai O(n^3)**
terhadap jumlah sampel, dan kebutuhan memori untuk matriks kernel juga kuadratik. Melatihnya pada
seluruh 96.146 baris di Google Colab (CPU standar) tidak realistis - bisa memakan berjam-jam untuk
satu konfigurasi saja, apalagi untuk sebuah grid.

Karena itu **eksperimen perbandingan kernel dan grid hyperparameter dijalankan pada subsample
stratified** (proporsi kelas dipertahankan persis seperti data penuh). Ini adalah **keterbatasan
metodologis yang disampaikan secara terbuka**, bukan disembunyikan. Model final (LinearSVC) tetap
dilatih pada data penuh karena kompleksitasnya mendekati linear terhadap n. Kesimpulan yang ditarik
dari subsample bersifat **komparatif** (mengurutkan kernel/parameter relatif satu sama lain), bukan
klaim performa absolut.

---

## Daftar eksperimen

| No | Eksperimen | Yang dibuktikan |
|----|------------|-----------------|
| 1 | Perbandingan kernel (linear, RBF, poly-2, poly-3, sigmoid) | Ruang hyperplane mana yang terbaik |
| 2 | Grid C (linear) dan C x gamma (RBF) | Nilai parameter mana yang optimal |
| 3 | Analisis margin dan support vector | **Inti jawaban**: hyperplane mana yang dipilih dan mengapa |
| 4 | Visualisasi hyperplane (2 fitur, PCA, linear vs RBF) | Bukti visual bentuk batas keputusan |
| 5 | Interpretasi vektor bobot w | Arti hyperplane secara klinis |
| 6 | Uji signifikansi linear vs RBF | Apakah selisihnya nyata secara statistik |
| 7 | Kalibrasi probabilitas | Justifikasi CalibratedClassifierCV |


In [ ]:
# ============================================================
# CELL 1: Instalasi Library
# ============================================================
!pip install -q pandas numpy matplotlib seaborn scikit-learn imbalanced-learn statsmodels kagglehub

In [ ]:
# ============================================================
# CELL 2: Import & Konstanta Global
# ============================================================
import os, json, time, math, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, StratifiedShuffleSplit,
    RepeatedStratifiedKFold, cross_validate, cross_val_predict,
    learning_curve, validation_curve, GridSearchCV, RandomizedSearchCV
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, confusion_matrix, classification_report,
    brier_score_loss
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

SELECTED_FEATURES = ['age', 'bmi', 'hypertension', 'HbA1c_level', 'blood_glucose_level']
FEATURE_LABELS    = ['Usia', 'BMI', 'Hipertensi', 'HbA1c', 'Kadar Glukosa']
TARGET            = 'diabetes'

WARNA_MODEL = {'Random Forest': '#3498db', 'KNN': '#e74c3c', 'SVM (Linear)': '#2ecc71'}
WARNA_AKSEN = '#f39c12'

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25
sns.set_style('whitegrid')

# --- Folder output -------------------------------------------------------
# Set PAKAI_DRIVE = True bila ingin hasil tersimpan permanen di Google Drive
# (WAJIB True kalau ingin notebook 06 membaca hasil notebook 01-05).
PAKAI_DRIVE = False

if PAKAI_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/DiaPredict_Revisi'
else:
    OUTPUT_DIR = '/content/hasil_revisi'

for sub in ['', '/tabel', '/gambar', '/json']:
    os.makedirs(OUTPUT_DIR + sub, exist_ok=True)

print(f'Folder output : {OUTPUT_DIR}')
print(f'Fitur         : {SELECTED_FEATURES}')

In [ ]:
# ============================================================
# CELL 3: Fungsi Utilitas Penyimpanan Hasil
# ============================================================
def simpan_tabel(df, nama, tampilkan=True):
    """Simpan DataFrame ke CSV di OUTPUT_DIR/tabel dan tampilkan."""
    path = f'{OUTPUT_DIR}/tabel/{nama}.csv'
    df.to_csv(path, index=False)
    print(f'[TABEL DISIMPAN] {path}')
    if tampilkan:
        display(df)
    return df

def simpan_json(obj, nama):
    """Simpan dict/list hasil eksperimen ke JSON (dipakai notebook 06 & website)."""
    path = f'{OUTPUT_DIR}/json/{nama}.json'
    def _konversi(o):
        if isinstance(o, (np.integer,)):  return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, (np.ndarray,)):  return o.tolist()
        if isinstance(o, (np.bool_,)):    return bool(o)
        return str(o)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=_konversi)
    print(f'[JSON DISIMPAN] {path}')
    return obj

def simpan_gambar(nama, fig=None, dpi=150):
    """Simpan figure matplotlib aktif ke OUTPUT_DIR/gambar."""
    path = f'{OUTPUT_DIR}/gambar/{nama}.png'
    (fig or plt).savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'[GAMBAR DISIMPAN] {path}')
    return path

def garis(judul='', lebar=70):
    print('=' * lebar)
    if judul:
        print(f'  {judul}')
        print('=' * lebar)

In [ ]:
# ============================================================
# CELL 4: Load Dataset + Cleaning + Winsorization
# (Identik dengan pipeline notebook V2 agar hasil dapat dibandingkan)
# ============================================================
import kagglehub

def muat_dan_bersihkan_data(verbose=True):
    path = kagglehub.dataset_download('iammustafatz/diabetes-prediction-dataset')
    csv_file = os.path.join(path, 'diabetes_prediction_dataset.csv')
    df_raw = pd.read_csv(csv_file)

    # 1) Hapus duplikat pada dataset penuh (SAMA seperti V2 -> sisa 96.146 baris)
    df = df_raw.drop_duplicates().reset_index(drop=True)

    # 2) Ambil 5 fitur terpilih + target
    df = df[SELECTED_FEATURES + [TARGET]].copy()

    # 3) Winsorization (capping IQR) hanya untuk fitur numerik non-biner
    numeric_feats = [f for f in SELECTED_FEATURES if df[f].nunique() > 2]
    ringkas = []
    for feat in numeric_feats:
        Q1, Q3 = df[feat].quantile(0.25), df[feat].quantile(0.75)
        IQR = Q3 - Q1
        low, up = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        n_cap = int(((df[feat] < low) | (df[feat] > up)).sum())
        df[feat] = df[feat].clip(lower=low, upper=up)
        ringkas.append({'fitur': feat, 'batas_bawah': low, 'batas_atas': up, 'n_dicapping': n_cap})

    if verbose:
        garis('DATA SIAP PAKAI')
        print(f'Baris (setelah hapus duplikat) : {len(df):,}')
        print(f'Distribusi kelas               : '
              f'{(df[TARGET]==0).sum():,} sehat / {(df[TARGET]==1).sum():,} diabetes '
              f'({df[TARGET].mean()*100:.2f}% positif)')
        display(pd.DataFrame(ringkas))
    return df

df_clean = muat_dan_bersihkan_data()
X_all = df_clean[SELECTED_FEATURES].copy()
y_all = df_clean[TARGET].copy()

In [ ]:
# ============================================================
# CELL 5: Pabrik Pipeline Model (anti data leakage)
# Urutan: StandardScaler -> SMOTE -> Classifier (imblearn Pipeline,
# sehingga SMOTE HANYA aktif saat fit, tidak saat predict/validasi)
# ============================================================

# Hyperparameter terbaik hasil tuning notebook V2 (baseline pembanding)
PARAM_RF_V2  = dict(n_estimators=200, max_depth=10, min_samples_split=5,
                    min_samples_leaf=4, max_features='log2', criterion='entropy',
                    class_weight='balanced')
PARAM_KNN_V2 = dict(n_neighbors=21, weights='uniform', metric='euclidean', leaf_size=20)
PARAM_SVM_V2 = dict(C=0.1, max_iter=3000)

def buat_pipeline_rf(pakai_smote=True, **params):
    p = {**PARAM_RF_V2, **params}
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **p)))
    return ImbPipeline(langkah)

def buat_pipeline_knn(pakai_smote=True, **params):
    p = {**PARAM_KNN_V2, **params}
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', KNeighborsClassifier(n_jobs=-1, **p)))
    return ImbPipeline(langkah)

def buat_pipeline_svm(pakai_smote=True, kernel='linear', C=0.1, gamma='scale',
                      degree=3, max_iter=3000, kalibrasi='sigmoid'):
    """kernel='linear' -> LinearSVC (cepat). Kernel lain -> SVC."""
    if kernel == 'linear':
        base = LinearSVC(C=C, max_iter=max_iter, class_weight='balanced',
                         dual=False, random_state=RANDOM_STATE)
    else:
        base = SVC(kernel=kernel, C=C, gamma=gamma, degree=degree,
                   class_weight='balanced', random_state=RANDOM_STATE)
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', CalibratedClassifierCV(base, cv=3, method=kalibrasi)))
    return ImbPipeline(langkah)

PABRIK_MODEL = {
    'Random Forest': buat_pipeline_rf,
    'KNN'          : buat_pipeline_knn,
    'SVM (Linear)' : buat_pipeline_svm,
}

In [ ]:
# ============================================================
# CELL 6: Fungsi Evaluasi Standar (dipakai seluruh notebook)
# ============================================================
def threshold_youden(y_true, y_proba):
    fpr, tpr, thr = roc_curve(y_true, y_proba)
    return float(thr[np.argmax(tpr - fpr)])

def hitung_metrik(y_true, y_pred, y_proba=None):
    hasil = {
        'accuracy' : accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall'   : recall_score(y_true, y_pred, zero_division=0),
        'f1'       : f1_score(y_true, y_pred, zero_division=0),
    }
    if y_proba is not None:
        hasil['roc_auc']  = roc_auc_score(y_true, y_proba)
        hasil['ap_score'] = average_precision_score(y_true, y_proba)
        hasil['brier']    = brier_score_loss(y_true, y_proba)
    return hasil

def evaluasi_holdout(model, X_tr, y_tr, X_te, y_te, tuning_threshold=True):
    """Fit -> prediksi -> metrik pada threshold 0.5 dan threshold Youden."""
    t0 = time.time(); model.fit(X_tr, y_tr); waktu_latih = time.time() - t0
    t0 = time.time(); y_proba = model.predict_proba(X_te)[:, 1]; waktu_infer = time.time() - t0

    thr = threshold_youden(y_te, y_proba) if tuning_threshold else 0.5
    m_def  = hitung_metrik(y_te, (y_proba >= 0.5).astype(int), y_proba)
    m_tune = hitung_metrik(y_te, (y_proba >= thr).astype(int), y_proba)
    return {
        'threshold': thr,
        'waktu_latih_s': waktu_latih,
        'waktu_infer_ms': waktu_infer * 1000,
        **{f'{k}_default': v for k, v in m_def.items()},
        **{f'{k}_tuned'  : v for k, v in m_tune.items()},
    }

def ci95_proporsi(p, n):
    """Confidence interval 95% (Wald) untuk metrik berbasis proporsi (mis. recall)."""
    if n == 0: return (np.nan, np.nan, np.nan)
    se = math.sqrt(max(p * (1 - p), 1e-12) / n)
    return (p - 1.96 * se, p + 1.96 * se, 1.96 * se)

---

## Persiapan eksperimen: subsample stratified dan estimasi biaya komputasi

Semua eksperimen di bawah memakai **subsample stratified** dengan ukuran berbeda sesuai berat
komputasinya, lalu dibagi **80:20 stratified** (rasio ini sendiri dijustifikasi di notebook `01`).
Data penuh tetap dipakai untuk model linear (LinearSVC) yang biayanya mendekati linear terhadap n.


In [ ]:
# ============================================================
# CELL 7: Konstanta Eksperimen SVM, Subsample Stratified, dan Split 80:20
# ============================================================
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from scipy import stats

MODE_CEPAT = True   # True = subsample (untuk Colab CPU). False = ukuran lebih besar.

# Ukuran subsample. SVC berkernel berkompleksitas O(n^2)-O(n^3) sehingga
# tidak realistis dilatih pada 96.146 baris di Colab CPU standar.
N_SUBSAMPLE_KERNEL = 20000 if MODE_CEPAT else 40000   # Eksperimen 1 (perbandingan kernel)
N_SUBSAMPLE_GRID   = 10000 if MODE_CEPAT else 20000   # Eksperimen 2 & 3 (grid C, margin)
N_SUBSAMPLE_SV     = 5000                             # jumlah support vector (SVC linear/RBF)
N_SUBSAMPLE_PLOT   = 3000                             # visualisasi hyperplane
N_SUBSAMPLE_PERM   = 5000                             # permutation importance

GRID_C       = [0.001, 0.01, 0.1, 1, 10, 100]
GRID_C_RBF   = [0.01, 0.1, 1, 10]
GRID_GAMMA   = ['scale', 0.01, 0.1, 1]

# Toleransi recall saat memilih C: ambil margin TERLEBAR di antara kandidat yang
# recall-nya masih dalam 1 poin persen dari recall terbaik (aturan eksplisit,
# bukan pilihan subjektif).
TOLERANSI_RECALL = 0.01

def ambil_subsample(X, y, n, seed=RANDOM_STATE):
    if n >= len(X):
        return X, y
    sss = StratifiedShuffleSplit(n_splits=1, train_size=n, random_state=seed)
    idx, _ = next(sss.split(X, y))
    return X.iloc[idx], y.iloc[idx]

def split_8020(X, y):
    """Split 80:20 stratified (rasio hasil justifikasi notebook 01)."""
    return train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

def siapkan_scale_smote(X_tr, y_tr, pakai_smote=True):
    """Preprocessing manual (scaler -> SMOTE) untuk analisis yang butuh akses
    langsung ke objek SVM (coef_, support_vectors_) di luar pipeline."""
    sc = StandardScaler().fit(X_tr)
    Xs = sc.transform(X_tr)
    ys = np.asarray(y_tr)
    if pakai_smote:
        Xs, ys = SMOTE(random_state=RANDOM_STATE).fit_resample(Xs, ys)
    return sc, Xs, ys

def evaluasi_decision(model, X_tr, y_tr, X_te, y_te):
    """Evaluasi TANPA kalibrasi: memakai decision_function (jarak bertanda ke
    hyperplane). Dipakai pada eksperimen grid yang berat, karena kalibrasi
    CalibratedClassifierCV(cv=3) menambah 3x biaya fit. Threshold tetap dipilih
    dengan kriteria Youden, hanya saja pada skala skor mentah, bukan probabilitas.
    ROC-AUC tidak terpengaruh karena bersifat invarian terhadap transformasi monoton."""
    t0 = time.time(); model.fit(X_tr, y_tr); waktu_latih = time.time() - t0
    t0 = time.time(); skor = model.decision_function(X_te); waktu_infer = time.time() - t0
    auc = roc_auc_score(y_te, skor)
    thr = threshold_youden(y_te, skor)
    m = hitung_metrik(y_te, (skor >= thr).astype(int))
    return {'threshold': thr, 'roc_auc': auc,
            'waktu_latih_s': waktu_latih, 'waktu_infer_ms': waktu_infer * 1000, **m}

def pipeline_linear_mentah(C, max_iter=5000, pakai_smote=True):
    """LinearSVC tanpa kalibrasi (untuk grid/margin). max_iter dinaikkan agar
    konfigurasi C kecil tetap konvergen."""
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', LinearSVC(C=C, max_iter=max_iter, class_weight='balanced',
                                     dual=False, random_state=RANDOM_STATE)))
    return ImbPipeline(langkah)

def pipeline_svc_mentah(kernel, C=1.0, gamma='scale', degree=3, pakai_smote=True):
    """SVC berkernel tanpa kalibrasi (untuk grid/uji signifikansi)."""
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', SVC(kernel=kernel, C=C, gamma=gamma, degree=degree,
                               class_weight='balanced', cache_size=500,
                               random_state=RANDOM_STATE)))
    return ImbPipeline(langkah)

# ---------------- Set data tiap eksperimen ----------------
X_kern, y_kern = ambil_subsample(X_all, y_all, N_SUBSAMPLE_KERNEL)
Xk_tr, Xk_te, yk_tr, yk_te = split_8020(X_kern, y_kern)

X_grid, y_grid = ambil_subsample(X_all, y_all, N_SUBSAMPLE_GRID)
Xg_tr, Xg_te, yg_tr, yg_te = split_8020(X_grid, y_grid)

X_sv, y_sv = ambil_subsample(X_all, y_all, N_SUBSAMPLE_SV)
Xs_tr, Xs_te, ys_tr, ys_te = split_8020(X_sv, y_sv)

# Data penuh: hanya untuk LinearSVC (biaya mendekati linear terhadap n)
Xf_tr, Xf_te, yf_tr, yf_te = split_8020(X_all, y_all)

garis('KONFIGURASI EKSPERIMEN HYPERPLANE SVM')
info_data = pd.DataFrame([
    {'eksperimen': '1. Perbandingan kernel', 'n_total': len(X_kern),
     'n_latih': len(Xk_tr), 'n_uji': len(Xk_te),
     'positif_%': round(float(y_kern.mean()) * 100, 2), 'estimasi_waktu': '8-25 menit'},
    {'eksperimen': '2. Grid C linear', 'n_total': len(X_grid),
     'n_latih': len(Xg_tr), 'n_uji': len(Xg_te),
     'positif_%': round(float(y_grid.mean()) * 100, 2), 'estimasi_waktu': '< 1 menit'},
    {'eksperimen': '2b. Grid C x gamma RBF', 'n_total': len(X_sv),
     'n_latih': len(Xs_tr), 'n_uji': len(Xs_te),
     'positif_%': round(float(y_sv.mean()) * 100, 2), 'estimasi_waktu': '3-8 menit'},
    {'eksperimen': '3. Margin & support vector', 'n_total': len(X_grid),
     'n_latih': len(Xg_tr), 'n_uji': len(Xg_te),
     'positif_%': round(float(y_grid.mean()) * 100, 2), 'estimasi_waktu': '2-6 menit'},
    {'eksperimen': '4. Visualisasi hyperplane', 'n_total': N_SUBSAMPLE_PLOT,
     'n_latih': N_SUBSAMPLE_PLOT, 'n_uji': 0,
     'positif_%': round(float(y_all.mean()) * 100, 2), 'estimasi_waktu': '< 1 menit'},
    {'eksperimen': '5. Bobot w (data penuh)', 'n_total': len(X_all),
     'n_latih': len(Xf_tr), 'n_uji': len(Xf_te),
     'positif_%': round(float(y_all.mean()) * 100, 2), 'estimasi_waktu': '1-3 menit'},
    {'eksperimen': '6. Uji linear vs RBF (5-fold)', 'n_total': len(X_sv),
     'n_latih': int(len(X_sv) * 0.8), 'n_uji': int(len(X_sv) * 0.2),
     'positif_%': round(float(y_sv.mean()) * 100, 2), 'estimasi_waktu': '2-5 menit'},
    {'eksperimen': '7. Kalibrasi probabilitas', 'n_total': len(X_kern),
     'n_latih': len(Xk_tr), 'n_uji': len(Xk_te),
     'positif_%': round(float(y_kern.mean()) * 100, 2), 'estimasi_waktu': '1-2 menit'},
])
display(info_data)
print(f'Proporsi positif data penuh   : {y_all.mean()*100:.2f}%  (subsample stratified -> proporsi dipertahankan)')
print(f'Baseline konfigurasi V2       : LinearSVC(C={PARAM_SVM_V2["C"]}, max_iter={PARAM_SVM_V2["max_iter"]}) + CalibratedClassifierCV(cv=3)')
print('Total estimasi waktu          : sekitar 20-50 menit pada Colab CPU standar.')

---

# EKSPERIMEN 1 - Perbandingan Kernel

**Pertanyaan:** di ruang mana hyperplane sebaiknya dicari?

Setiap kandidat dibungkus dalam pipeline yang **sama persis** dengan model V2
(`StandardScaler -> SMOTE -> CalibratedClassifierCV(cv=3)`), memakai `class_weight='balanced'`,
dan dievaluasi pada test set yang sama dengan **threshold Youden**. Dengan begitu satu-satunya
yang berbeda antar-baris adalah **kernel** (ruang pencarian hyperplane).

Kandidat: `LinearSVC`, `SVC(kernel='linear')`, `SVC(kernel='rbf')`, `SVC(kernel='poly', degree=2)`,
`SVC(kernel='poly', degree=3)`, `SVC(kernel='sigmoid')`. Untuk keadilan perbandingan, seluruh
kandidat diuji pada `C=1.0` (default scikit-learn), ditambah satu baris `LinearSVC(C=0.1)` sebagai
konfigurasi V2 yang ada sekarang. Nilai C dioptimalkan tersendiri pada Eksperimen 2.


In [ ]:
# ============================================================
# CELL 8: Eksperimen 1 - Perbandingan Kernel SVM
# ============================================================
def buat_pipeline_svc_kalibrasi(kernel, C=1.0, gamma='scale', degree=3,
                                pakai_smote=True, kalibrasi='sigmoid'):
    """Sama strukturnya dengan buat_pipeline_svm (CELL 5), tetapi selalu memakai SVC
    sehingga kernel 'linear' pun dijalankan lewat SVC (bukan LinearSVC). Ini penting
    agar perbandingan kernel adil: LinearSVC dan SVC(kernel='linear') memakai solver
    dan formulasi loss yang berbeda (squared_hinge/primal vs hinge/dual)."""
    base = SVC(kernel=kernel, C=C, gamma=gamma, degree=degree,
               class_weight='balanced', cache_size=500, random_state=RANDOM_STATE)
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', CalibratedClassifierCV(base, cv=3, method=kalibrasi)))
    return ImbPipeline(langkah)

KANDIDAT_KERNEL = [
    {'nama': 'LinearSVC (C=0.1) [V2]', 'tipe': 'linearsvc', 'kernel': 'linear',
     'C': 0.1, 'gamma': '-', 'degree': '-'},
    {'nama': 'LinearSVC (C=1.0)', 'tipe': 'linearsvc', 'kernel': 'linear',
     'C': 1.0, 'gamma': '-', 'degree': '-'},
    {'nama': 'SVC linear (C=1.0)', 'tipe': 'svc', 'kernel': 'linear',
     'C': 1.0, 'gamma': '-', 'degree': '-'},
    {'nama': 'SVC RBF (C=1.0)', 'tipe': 'svc', 'kernel': 'rbf',
     'C': 1.0, 'gamma': 'scale', 'degree': '-'},
    {'nama': 'SVC Poly d=2 (C=1.0)', 'tipe': 'svc', 'kernel': 'poly',
     'C': 1.0, 'gamma': 'scale', 'degree': 2},
    {'nama': 'SVC Poly d=3 (C=1.0)', 'tipe': 'svc', 'kernel': 'poly',
     'C': 1.0, 'gamma': 'scale', 'degree': 3},
    {'nama': 'SVC Sigmoid (C=1.0)', 'tipe': 'svc', 'kernel': 'sigmoid',
     'C': 1.0, 'gamma': 'scale', 'degree': '-'},
]

garis('EKSPERIMEN 1: PERBANDINGAN KERNEL SVM')
print(f'Data   : {len(Xk_tr):,} latih / {len(Xk_te):,} uji (subsample stratified)')
print(f'Pipeline: StandardScaler -> SMOTE -> CalibratedClassifierCV(cv=3, sigmoid)')
print(f'Metrik  : threshold Youden (memaksimalkan sensitivitas - spesifisitas)')
print(f'Estimasi: 8-25 menit. Kernel yang gagal/terlalu lama ditangkap try/except.')
print('-' * 70)

hasil_kernel = []
t_mulai_total = time.time()

for i, kand in enumerate(KANDIDAT_KERNEL, 1):
    print(f'[{i}/{len(KANDIDAT_KERNEL)}] {kand["nama"]} ... ', end='')
    try:
        deg = kand['degree'] if isinstance(kand['degree'], int) else 3
        if kand['tipe'] == 'linearsvc':
            model = buat_pipeline_svm(kernel='linear', C=kand['C'],
                                      max_iter=PARAM_SVM_V2['max_iter'])
        else:
            model = buat_pipeline_svc_kalibrasi(kernel=kand['kernel'], C=kand['C'],
                                                gamma=kand['gamma'] if kand['gamma'] != '-' else 'scale',
                                                degree=deg)
        r = evaluasi_holdout(model, Xk_tr, yk_tr, Xk_te, yk_te)
        hasil_kernel.append({
            'kandidat'      : kand['nama'],
            'kernel'        : kand['kernel'],
            'implementasi'  : 'LinearSVC' if kand['tipe'] == 'linearsvc' else 'SVC',
            'C'             : kand['C'],
            'gamma'         : kand['gamma'],
            'degree'        : kand['degree'],
            'recall'        : r['recall_tuned'],
            'precision'     : r['precision_tuned'],
            'f1'            : r['f1_tuned'],
            'roc_auc'       : r['roc_auc_tuned'],
            'accuracy'      : r['accuracy_tuned'],
            'brier'         : r['brier_tuned'],
            'threshold'     : r['threshold'],
            'waktu_latih_s' : r['waktu_latih_s'],
            'waktu_infer_ms': r['waktu_infer_ms'],
            'status'        : 'OK',
        })
        print(f'OK | recall={r["recall_tuned"]:.4f} AUC={r["roc_auc_tuned"]:.4f} '
              f'latih={r["waktu_latih_s"]:.1f}s')
    except Exception as e:
        hasil_kernel.append({
            'kandidat': kand['nama'], 'kernel': kand['kernel'],
            'implementasi': 'LinearSVC' if kand['tipe'] == 'linearsvc' else 'SVC',
            'C': kand['C'], 'gamma': kand['gamma'], 'degree': kand['degree'],
            'recall': np.nan, 'precision': np.nan, 'f1': np.nan, 'roc_auc': np.nan,
            'accuracy': np.nan, 'brier': np.nan, 'threshold': np.nan,
            'waktu_latih_s': np.nan, 'waktu_infer_ms': np.nan,
            'status': f'GAGAL: {type(e).__name__}',
        })
        print(f'GAGAL ({type(e).__name__}: {str(e)[:60]}) -> dilewati, notebook lanjut')

print('-' * 70)
print(f'Total waktu Eksperimen 1: {(time.time() - t_mulai_total)/60:.1f} menit')

tabel_perbandingan_kernel = pd.DataFrame(hasil_kernel).round(5)
simpan_tabel(tabel_perbandingan_kernel, 'tabel_perbandingan_kernel')

# --- Kesimpulan angka ---
ok = tabel_perbandingan_kernel[tabel_perbandingan_kernel['status'] == 'OK'].copy()
baris_auc_terbaik    = ok.loc[ok['roc_auc'].idxmax()]
baris_recall_terbaik = ok.loc[ok['recall'].idxmax()]
baris_linear         = ok[ok['kernel'] == 'linear'].sort_values('roc_auc', ascending=False).iloc[0]
baris_rbf            = ok[ok['kernel'] == 'rbf'].iloc[0] if (ok['kernel'] == 'rbf').any() else None

garis('KESIMPULAN EKSPERIMEN 1')
print(f'AUC tertinggi        : {baris_auc_terbaik["kandidat"]} (AUC={baris_auc_terbaik["roc_auc"]:.4f})')
print(f'Recall tertinggi     : {baris_recall_terbaik["kandidat"]} (recall={baris_recall_terbaik["recall"]:.4f})')
print(f'Kernel linear terbaik: {baris_linear["kandidat"]} '
      f'(AUC={baris_linear["roc_auc"]:.4f}, recall={baris_linear["recall"]:.4f}, '
      f'latih={baris_linear["waktu_latih_s"]:.1f}s)')
if baris_rbf is not None:
    d_auc = baris_rbf['roc_auc'] - baris_linear['roc_auc']
    rasio_waktu = baris_rbf['waktu_latih_s'] / max(baris_linear['waktu_latih_s'], 1e-9)
    print(f'Kernel RBF           : AUC={baris_rbf["roc_auc"]:.4f}, recall={baris_rbf["recall"]:.4f}, '
          f'latih={baris_rbf["waktu_latih_s"]:.1f}s')
    print(f'Selisih AUC RBF-linear : {d_auc:+.4f}  |  RBF {rasio_waktu:.1f}x lebih lambat dilatih')
print()
print('Interpretasi: bila selisih AUC antar-kernel berada pada orde 0,00x sementara biaya')
print('komputasi kernel non-linear berkali-kali lipat, maka memindahkan hyperplane ke ruang')
print('berdimensi tinggi TIDAK memberi keuntungan yang sepadan pada dataset ini.')

In [ ]:
# ============================================================
# CELL 9: Visualisasi Perbandingan Kernel (metrik + waktu latih)
# ============================================================
ok_plot = tabel_perbandingan_kernel[tabel_perbandingan_kernel['status'] == 'OK'].copy()
metrik_plot = ['recall', 'precision', 'f1', 'roc_auc']
label_metrik = ['Recall', 'Precision', 'F1-Score', 'ROC-AUC']
warna_metrik = ['#2ecc71', '#3498db', '#9b59b6', '#f39c12']

fig, axes = plt.subplots(1, 2, figsize=(18, 6.5))

# (a) Bar berkelompok metrik per kernel
x = np.arange(len(ok_plot))
lebar = 0.2
for j, (m, lab, w) in enumerate(zip(metrik_plot, label_metrik, warna_metrik)):
    axes[0].bar(x + (j - 1.5) * lebar, ok_plot[m].values, lebar, label=lab, color=w, edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(ok_plot['kandidat'], rotation=30, ha='right', fontsize=9)
axes[0].set_ylabel('Nilai metrik')
axes[0].set_title('(a) Performa per Kernel (threshold Youden)')
axes[0].set_ylim(0, 1.05)
axes[0].legend(ncol=4, fontsize=9, loc='upper center')
for xi, v in zip(x, ok_plot['recall'].values):
    axes[0].text(xi - 1.5 * lebar, v + 0.015, f'{v:.3f}', ha='center', fontsize=7.5, rotation=90)

# (b) Waktu latih (skala log)
warna_bar = [WARNA_MODEL['SVM (Linear)'] if k == 'linear' else '#95a5a6'
             for k in ok_plot['kernel']]
axes[1].bar(x, ok_plot['waktu_latih_s'].values, color=warna_bar, edgecolor='white')
axes[1].set_yscale('log')
axes[1].set_xticks(x)
axes[1].set_xticklabels(ok_plot['kandidat'], rotation=30, ha='right', fontsize=9)
axes[1].set_ylabel('Waktu latih (detik, skala log)')
axes[1].set_title('(b) Biaya Komputasi Pelatihan - hijau = kernel linear')
for xi, v in zip(x, ok_plot['waktu_latih_s'].values):
    axes[1].text(xi, v * 1.12, f'{v:.1f}s', ha='center', fontsize=8.5)

plt.suptitle('Eksperimen 1 - Perbandingan Kernel SVM: Performa vs Biaya Komputasi',
             fontsize=14, y=1.02)
plt.tight_layout()
simpan_gambar('svm_perbandingan_kernel')
plt.show()

print('Bacaan grafik: panel (a) menunjukkan metrik antar-kernel praktis berimpit,')
print('sedangkan panel (b) berskala LOGARITMIK - artinya perbedaan biaya latih')
print('mencapai orde puluhan hingga ratusan kali lipat.')

---

# EKSPERIMEN 2 - Grid Parameter C dan Gamma

**Pertanyaan:** setelah ruangnya dipilih, hyperplane yang mana (parameter mana) di ruang itu?

- **Kernel linear:** sapuan `C` pada [0.001, 0.01, 0.1, 1, 10, 100]. C mengatur trade-off
  margin lebar vs kesalahan klasifikasi.
- **Kernel RBF:** grid `C` x `gamma`. Gamma mengatur seberapa "lokal" pengaruh tiap support vector -
  gamma besar membuat batas keputusan sangat berkelok (risiko overfitting).

Untuk grid ini kalibrasi **tidak** dipakai (cukup `decision_function`), karena
`CalibratedClassifierCV(cv=3)` melipatgandakan biaya fit 3x sementara ROC-AUC bersifat invarian
terhadap transformasi monoton seperti kalibrasi sigmoid.


In [ ]:
# ============================================================
# CELL 10: Eksperimen 2a - Sapuan Parameter C untuk Kernel Linear
# ============================================================
garis('EKSPERIMEN 2a: SAPUAN C UNTUK KERNEL LINEAR')
print(f'Data     : {len(Xg_tr):,} latih / {len(Xg_te):,} uji')
print(f'Grid C   : {GRID_C}')
print('Estimasi : < 1 menit')
print('-' * 70)

hasil_C = []
for C in GRID_C:
    t0 = time.time()
    try:
        r = evaluasi_decision(pipeline_linear_mentah(C), Xg_tr, yg_tr, Xg_te, yg_te)
        hasil_C.append({'kernel': 'linear', 'C': C, 'gamma': '-',
                        'recall': r['recall'], 'precision': r['precision'],
                        'f1': r['f1'], 'roc_auc': r['roc_auc'],
                        'accuracy': r['accuracy'], 'threshold': r['threshold'],
                        'waktu_latih_s': r['waktu_latih_s'], 'status': 'OK'})
        print(f'C={C:<8} recall={r["recall"]:.4f}  precision={r["precision"]:.4f}  '
              f'F1={r["f1"]:.4f}  AUC={r["roc_auc"]:.4f}  ({time.time()-t0:.1f}s)')
    except Exception as e:
        hasil_C.append({'kernel': 'linear', 'C': C, 'gamma': '-', 'recall': np.nan,
                        'precision': np.nan, 'f1': np.nan, 'roc_auc': np.nan,
                        'accuracy': np.nan, 'threshold': np.nan,
                        'waktu_latih_s': np.nan, 'status': f'GAGAL: {type(e).__name__}'})
        print(f'C={C:<8} GAGAL ({type(e).__name__}) -> dilewati')

tabel_grid_C_linear = pd.DataFrame(hasil_C).round(5)
simpan_tabel(tabel_grid_C_linear, 'tabel_grid_C_linear')

# --- Kurva metrik vs C (sumbu-x logaritmik) ---
ok_C = tabel_grid_C_linear[tabel_grid_C_linear['status'] == 'OK']
plt.figure(figsize=(12, 6))
plt.semilogx(ok_C['C'], ok_C['recall'], 'o-', lw=2.2, ms=8,
             color=WARNA_MODEL['SVM (Linear)'], label='Recall (sensitivitas)')
plt.semilogx(ok_C['C'], ok_C['f1'], 's-', lw=2.2, ms=7, color='#9b59b6', label='F1-Score')
plt.semilogx(ok_C['C'], ok_C['roc_auc'], '^-', lw=2.2, ms=7, color=WARNA_AKSEN, label='ROC-AUC')
plt.semilogx(ok_C['C'], ok_C['precision'], 'd--', lw=1.6, ms=6, color='#3498db',
             alpha=0.75, label='Precision')
plt.axvline(PARAM_SVM_V2['C'], color='#e74c3c', ls=':', lw=2,
            label=f'C = {PARAM_SVM_V2["C"]} (hasil tuning V2)')
plt.xlabel('Parameter C (skala logaritmik) - kiri: margin lebar, kanan: margin sempit')
plt.ylabel('Nilai metrik pada test set')
plt.title('Eksperimen 2a - Pengaruh Parameter C terhadap Performa SVM Linear')
plt.legend(loc='best', fontsize=10)
plt.ylim(0, 1.05)
plt.tight_layout()
simpan_gambar('svm_kurva_C')
plt.show()

if len(ok_C):
    b_auc = ok_C.loc[ok_C['roc_auc'].idxmax()]
    b_rec = ok_C.loc[ok_C['recall'].idxmax()]
    rentang_auc = ok_C['roc_auc'].max() - ok_C['roc_auc'].min()
    garis('KESIMPULAN EKSPERIMEN 2a')
    print(f'AUC tertinggi    : C={b_auc["C"]} (AUC={b_auc["roc_auc"]:.4f})')
    print(f'Recall tertinggi : C={b_rec["C"]} (recall={b_rec["recall"]:.4f})')
    print(f'Rentang AUC pada seluruh grid C: {rentang_auc:.4f}')
    print('Kurva yang datar berarti performa TIDAK sensitif terhadap C. Dalam kondisi ini,')
    print('C kecil lebih dipilih karena menghasilkan margin lebih lebar (regularisasi lebih')
    print('kuat) tanpa mengorbankan performa - dibuktikan secara kuantitatif di Eksperimen 3.')

In [ ]:
# ============================================================
# CELL 11: Eksperimen 2b - Grid C x Gamma untuk Kernel RBF (heatmap recall)
# ============================================================
# Subsample lebih kecil dipakai di sini karena SVC-RBF berkompleksitas O(n^2)-O(n^3)
# dan grid ini berisi len(GRID_C_RBF) x len(GRID_GAMMA) kombinasi.
garis('EKSPERIMEN 2b: GRID C x GAMMA UNTUK KERNEL RBF')
print(f'Data     : {len(Xs_tr):,} latih / {len(Xs_te):,} uji (subsample lebih kecil, alasan biaya O(n^2)-O(n^3))')
print(f'Grid     : C={GRID_C_RBF} x gamma={GRID_GAMMA} -> {len(GRID_C_RBF)*len(GRID_GAMMA)} kombinasi')
print('Estimasi : 3-8 menit')
print('-' * 70)

hasil_rbf = []
t_mulai = time.time()
for i, (C, g) in enumerate(itertools.product(GRID_C_RBF, GRID_GAMMA), 1):
    try:
        r = evaluasi_decision(pipeline_svc_mentah('rbf', C=C, gamma=g),
                              Xs_tr, ys_tr, Xs_te, ys_te)
        hasil_rbf.append({'kernel': 'rbf', 'C': C, 'gamma': str(g),
                          'recall': r['recall'], 'precision': r['precision'],
                          'f1': r['f1'], 'roc_auc': r['roc_auc'],
                          'accuracy': r['accuracy'], 'threshold': r['threshold'],
                          'waktu_latih_s': r['waktu_latih_s'], 'status': 'OK'})
        print(f'[{i:2d}/{len(GRID_C_RBF)*len(GRID_GAMMA)}] C={C:<6} gamma={str(g):<6} '
              f'recall={r["recall"]:.4f} AUC={r["roc_auc"]:.4f} ({r["waktu_latih_s"]:.1f}s)')
    except Exception as e:
        hasil_rbf.append({'kernel': 'rbf', 'C': C, 'gamma': str(g), 'recall': np.nan,
                          'precision': np.nan, 'f1': np.nan, 'roc_auc': np.nan,
                          'accuracy': np.nan, 'threshold': np.nan, 'waktu_latih_s': np.nan,
                          'status': f'GAGAL: {type(e).__name__}'})
        print(f'[{i:2d}] C={C} gamma={g} GAGAL ({type(e).__name__}) -> dilewati')

print('-' * 70)
print(f'Total waktu Eksperimen 2b: {(time.time() - t_mulai)/60:.1f} menit')

tabel_rbf = pd.DataFrame(hasil_rbf).round(5)

# Gabungan grid linear + RBF (format panjang) sesuai kontrak nama file spec
tabel_grid_C_gamma = pd.concat([tabel_grid_C_linear, tabel_rbf], ignore_index=True)
simpan_tabel(tabel_grid_C_gamma, 'tabel_grid_C_gamma')

# --- Heatmap recall ---
piv_recall = tabel_rbf.pivot(index='C', columns='gamma', values='recall')
piv_auc    = tabel_rbf.pivot(index='C', columns='gamma', values='roc_auc')
urut_gamma = [str(g) for g in GRID_GAMMA]
piv_recall = piv_recall.reindex(columns=urut_gamma)
piv_auc    = piv_auc.reindex(columns=urut_gamma)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.8))
sns.heatmap(piv_recall, annot=True, fmt='.4f', cmap='YlGnBu', ax=axes[0],
            cbar_kws={'label': 'Recall'}, linewidths=0.5)
axes[0].set_title('(a) Recall - Grid C x Gamma (kernel RBF)')
axes[0].set_xlabel('gamma'); axes[0].set_ylabel('C')

sns.heatmap(piv_auc, annot=True, fmt='.4f', cmap='YlOrRd', ax=axes[1],
            cbar_kws={'label': 'ROC-AUC'}, linewidths=0.5)
axes[1].set_title('(b) ROC-AUC - Grid C x Gamma (kernel RBF)')
axes[1].set_xlabel('gamma'); axes[1].set_ylabel('C')

plt.suptitle('Eksperimen 2b - Pemetaan Parameter Kernel RBF', fontsize=14, y=1.02)
plt.tight_layout()
simpan_gambar('svm_heatmap_rbf')
plt.show()

ok_rbf = tabel_rbf[tabel_rbf['status'] == 'OK']
garis('KESIMPULAN EKSPERIMEN 2b')
if len(ok_rbf):
    best_rbf = ok_rbf.loc[ok_rbf['roc_auc'].idxmax()]
    best_lin_auc = float(ok_C['roc_auc'].max()) if len(ok_C) else float('nan')
    print(f'RBF terbaik      : C={best_rbf["C"]}, gamma={best_rbf["gamma"]} '
          f'-> AUC={best_rbf["roc_auc"]:.4f}, recall={best_rbf["recall"]:.4f}')
    print(f'Linear terbaik   : AUC={best_lin_auc:.4f} (Eksperimen 2a)')
    print(f'Selisih AUC (RBF terbaik - linear terbaik): {best_rbf["roc_auc"] - best_lin_auc:+.4f}')
    print(f'Waktu latih RBF terbaik: {best_rbf["waktu_latih_s"]:.1f}s pada {len(Xs_tr):,} sampel saja,')
    print(f'sedangkan kernel linear melatih {len(Xg_tr):,} sampel dalam '
          f'{float(ok_C["waktu_latih_s"].min()):.2f}s.')
    print()
    print('Catatan: gamma besar (>= 1) menaikkan kompleksitas batas keputusan namun tidak')
    print('menaikkan AUC secara berarti - indikasi bahwa struktur pemisah kelas pada 5 fitur')
    print('ini memang mendekati linear, bukan berbentuk kantong-kantong non-linear.')
else:
    print('Seluruh konfigurasi RBF gagal dijalankan - lihat kolom status pada tabel.')

---

# EKSPERIMEN 3 - Analisis Margin dan Support Vector (INTI JAWABAN)

Ini adalah bagian yang menjawab pertanyaan penguji **secara harfiah**: dari sekian banyak
hyperplane yang mungkin, **yang mana yang dipilih dan mengapa**.

Untuk tiap nilai C dihitung:

| Besaran | Rumus | Arti |
|---|---|---|
| $\lVert w \rVert$ | norma vektor bobot | seberapa "curam" fungsi keputusan |
| margin | $2 / \lVert w \rVert$ | lebar koridor pemisah antar kelas |
| $n_{SV}$ | jumlah support vector | banyaknya sampel yang menentukan hyperplane |
| rasio SV | $n_{SV}/n$ | proporsi data yang menempel di margin |

**Catatan implementasi (penting):** `LinearSVC` menyelesaikan masalah optimasi dalam bentuk
**primal** sehingga hanya menyimpan `coef_` dan **tidak memiliki atribut `support_vectors_`**.
Karena itu $\lVert w \rVert$ dan margin diambil dari `LinearSVC` (model yang benar-benar dipakai
sistem), sedangkan **jumlah support vector dihitung dengan `SVC(kernel='linear')`** yang
menyelesaikan bentuk **dual** dan menyimpan `n_support_`. SVC dijalankan pada subsample yang lebih
kecil karena biaya komputasinya kuadratik.

**Aturan pemilihan (ditetapkan di muka, bukan post-hoc):** dipilih hyperplane dengan
**margin terlebar** di antara kandidat yang recall-nya masih berada dalam **1 poin persen** dari
recall terbaik. Prinsipnya: dalam skrining medis sensitivitas tidak boleh dikorbankan, tetapi di
antara pilihan yang sama sensitifnya, margin terlebar berarti generalisasi terbaik.


In [ ]:
# ============================================================
# CELL 12: Eksperimen 3 - Margin, Norma Bobot, dan Jumlah Support Vector
# ============================================================
garis('EKSPERIMEN 3: ANALISIS MARGIN DAN SUPPORT VECTOR')
print(f'Norma bobot & margin : LinearSVC pada {len(Xg_tr):,} sampel latih (model yang dipakai sistem)')
print(f'Jumlah support vector: SVC(kernel="linear") pada {len(Xs_tr):,} sampel latih')
print('  -> LinearSVC memakai formulasi PRIMAL sehingga tidak menyimpan support_vectors_.')
print('     SVC memakai formulasi DUAL sehingga n_support_ tersedia.')
print(f'Grid C  : {GRID_C}')
print('Estimasi: 2-6 menit')
print('-' * 70)

# Preprocessing manual (scaler -> SMOTE) supaya objek SVM bisa diakses langsung
sc_m,  Xm_res,  ym_res  = siapkan_scale_smote(Xg_tr, yg_tr)   # untuk LinearSVC (margin)
sc_sv, Xsv_res, ysv_res = siapkan_scale_smote(Xs_tr, ys_tr)   # untuk SVC (support vector)
Xg_te_s = sc_m.transform(Xg_te)

baris_margin = []
for C in GRID_C:
    t0 = time.time()
    try:
        # (1) LinearSVC -> vektor bobot w, norma, lebar margin
        lin = LinearSVC(C=C, max_iter=5000, class_weight='balanced',
                        dual=False, random_state=RANDOM_STATE)
        lin.fit(Xm_res, ym_res)
        w = lin.coef_.ravel()
        norma_w = float(np.linalg.norm(w))
        margin  = 2.0 / norma_w

        skor = lin.decision_function(Xg_te_s)
        thr  = threshold_youden(yg_te, skor)
        met  = hitung_metrik(yg_te, (skor >= thr).astype(int))
        auc  = roc_auc_score(yg_te, skor)

        # (2) SVC(kernel='linear') -> jumlah support vector
        svc = SVC(kernel='linear', C=C, class_weight='balanced',
                  cache_size=500, random_state=RANDOM_STATE)
        svc.fit(Xsv_res, ysv_res)
        n_sv = int(svc.n_support_.sum())
        rasio_sv = n_sv / len(Xsv_res)
        norma_w_svc = float(np.linalg.norm(svc.coef_.ravel()))

        baris_margin.append({
            'C': C, 'norma_w': norma_w, 'margin_2_per_w': margin,
            'n_SV': n_sv, 'rasio_SV': rasio_sv, 'norma_w_svc': norma_w_svc,
            'recall': met['recall'], 'precision': met['precision'],
            'f1': met['f1'], 'roc_auc': auc, 'status': 'OK'})
        print(f'C={C:<8} ||w||={norma_w:7.4f}  margin={margin:8.4f}  '
              f'n_SV={n_sv:5d} ({rasio_sv*100:5.1f}%)  recall={met["recall"]:.4f}  '
              f'precision={met["precision"]:.4f}  ({time.time()-t0:.1f}s)')
    except Exception as e:
        baris_margin.append({'C': C, 'norma_w': np.nan, 'margin_2_per_w': np.nan,
                             'n_SV': np.nan, 'rasio_SV': np.nan, 'norma_w_svc': np.nan,
                             'recall': np.nan, 'precision': np.nan, 'f1': np.nan,
                             'roc_auc': np.nan, 'status': f'GAGAL: {type(e).__name__}'})
        print(f'C={C:<8} GAGAL ({type(e).__name__}) -> dilewati')

tabel_analisis_margin = pd.DataFrame(baris_margin).round(6)
simpan_tabel(tabel_analisis_margin, 'tabel_analisis_margin')

# --- Aturan pemilihan hyperplane: margin TERLEBAR di antara yang recall-nya masih tinggi ---
ok_m = tabel_analisis_margin[tabel_analisis_margin['status'] == 'OK'].copy()
recall_maks = float(ok_m['recall'].max())
kandidat_layak = ok_m[ok_m['recall'] >= recall_maks - TOLERANSI_RECALL].copy()
baris_terpilih = kandidat_layak.loc[kandidat_layak['margin_2_per_w'].idxmax()]

C_TERPILIH      = float(baris_terpilih['C'])
MARGIN_TERPILIH = float(baris_terpilih['margin_2_per_w'])
NORMA_W_TERPILIH= float(baris_terpilih['norma_w'])
NSV_TERPILIH    = int(baris_terpilih['n_SV'])
RASIO_SV_TERPILIH = float(baris_terpilih['rasio_SV'])

garis('KESIMPULAN EKSPERIMEN 3 - HYPERPLANE YANG DIPILIH')
print(f'Recall terbaik pada grid          : {recall_maks:.4f}')
print(f'Kandidat dalam toleransi {TOLERANSI_RECALL*100:.0f}% recall : C = {list(kandidat_layak["C"].values)}')
print(f'-> C TERPILIH                     : {C_TERPILIH}')
print(f'   ||w||                          : {NORMA_W_TERPILIH:.4f}')
print(f'   Lebar margin (2/||w||)         : {MARGIN_TERPILIH:.4f}')
print(f'   Jumlah support vector          : {NSV_TERPILIH:,} dari {len(Xsv_res):,} sampel '
      f'({RASIO_SV_TERPILIH*100:.1f}%)')
print(f'   Recall / Precision             : {baris_terpilih["recall"]:.4f} / {baris_terpilih["precision"]:.4f}')
print()
c_maks = ok_m.loc[ok_m['C'].idxmax()]
print(f'Sebagai pembanding, C={c_maks["C"]} menghasilkan margin {c_maks["margin_2_per_w"]:.4f} '
      f'({MARGIN_TERPILIH / max(c_maks["margin_2_per_w"], 1e-12):.1f}x lebih sempit)')
print(f'dengan recall {c_maks["recall"]:.4f} - artinya mengetatkan C hanya mempersempit margin')
print('tanpa imbalan sensitivitas. Inilah alasan empiris hyperplane bermargin lebar yang dipilih.')
print()
print(f'Konfigurasi V2 memakai C={PARAM_SVM_V2["C"]} -> '
      f'{"KONSISTEN dengan hasil eksperimen ini" if abs(C_TERPILIH - PARAM_SVM_V2["C"]) < 1e-9 else "berbeda, lihat tabel di atas"}')

In [ ]:
# ============================================================
# CELL 13: Dua Panel Bertumpuk - Lebar Margin dan Recall terhadap C
# ============================================================
# Catatan desain: grafik ini sengaja TIDAK memakai dua sumbu-y (twinx). Pada grafik
# dua sumbu, posisi relatif kedua kurva - termasuk titik potongnya - sepenuhnya
# ditentukan oleh pilihan rentang masing-masing sumbu, sehingga titik potong bisa
# digeser ke mana saja hanya dengan mengubah batas sumbu. Karena justru titik itulah
# yang menjadi inti argumen pemilihan hyperplane, dipakai dua panel bertumpuk yang
# BERBAGI sumbu-x nilai C: pembaca membandingkan secara vertikal pada C yang identik,
# dan tiap panel punya sumbu-y sendiri yang berdiri sendiri.
ok_m = tabel_analisis_margin[tabel_analisis_margin['status'] == 'OK'].copy()

# Rentang C yang memenuhi aturan pemilihan (recall dalam toleransi dari recall terbaik)
recall_maks_plot = float(ok_m['recall'].max())
layak_plot = ok_m[ok_m['recall'] >= recall_maks_plot - TOLERANSI_RECALL]
c_min_layak = float(layak_plot['C'].min())
c_maks_layak = float(layak_plot['C'].max())

fig, (ax_atas, ax_bawah) = plt.subplots(2, 1, figsize=(13, 9.5), sharex=True,
                                        gridspec_kw={'height_ratios': [1, 1]})

# ---------- Panel atas: lebar margin 2/||w|| ----------
ax_atas.plot(ok_m['C'], ok_m['margin_2_per_w'], 'o-', lw=2.6, ms=9,
             color=WARNA_MODEL['SVM (Linear)'], label='Lebar margin  2/||w||')
ax_atas.set_yscale('log')
ax_atas.set_ylabel('Lebar margin  2 / ||w||\n(skala log)')
ax_atas.set_title('Eksperimen 3 - Lebar Margin dan Sensitivitas terhadap Parameter C\n'
                  '(dua panel berbagi sumbu-x: bandingkan secara vertikal pada nilai C yang sama)',
                  fontsize=13)

# ---------- Panel bawah: recall dan precision ----------
ax_bawah.plot(ok_m['C'], ok_m['recall'], 's-', lw=2.4, ms=8,
              color='#e74c3c', label='Recall (sensitivitas)')
ax_bawah.plot(ok_m['C'], ok_m['precision'], 'd--', lw=1.8, ms=6,
              color='#3498db', alpha=0.85, label='Precision')
ax_bawah.axhline(recall_maks_plot - TOLERANSI_RECALL, color='#e74c3c', ls=':', lw=1.5,
                 alpha=0.8,
                 label=f'Batas toleransi recall ({recall_maks_plot - TOLERANSI_RECALL:.4f})')
ax_bawah.set_ylim(0, 1.05)
ax_bawah.set_ylabel('Recall / Precision')
ax_bawah.set_xscale('log')
ax_bawah.set_xlabel('Parameter C (skala logaritmik) - kiri: margin lebar, kanan: margin sempit')

# ---------- Penanda identik pada KEDUA panel ----------
for ax in (ax_atas, ax_bawah):
    ax.axvspan(c_min_layak, c_maks_layak, color=WARNA_AKSEN, alpha=0.12,
               label=f'Rentang C yang lolos aturan (recall >= terbaik - {TOLERANSI_RECALL*100:.0f} poin persen)')
    ax.axvline(C_TERPILIH, color=WARNA_AKSEN, ls='-', lw=2.5, alpha=0.9,
               label=f'C terpilih = {C_TERPILIH}')
    ax.set_xlim(min(ok_m['C']) / 3, max(ok_m['C']) * 3)

# Tandai titik-titik yang lolos aturan pada kedua panel
ax_atas.plot(layak_plot['C'], layak_plot['margin_2_per_w'], 'o', ms=14,
             mfc='none', mec=WARNA_AKSEN, mew=2.2)
ax_bawah.plot(layak_plot['C'], layak_plot['recall'], 's', ms=14,
              mfc='none', mec=WARNA_AKSEN, mew=2.2)

ax_atas.annotate(f'C terpilih = {C_TERPILIH}\nmargin = {MARGIN_TERPILIH:.3f} (TERLEBAR di rentang ini)\n'
                 f'n_SV = {NSV_TERPILIH:,} ({RASIO_SV_TERPILIH*100:.1f}%)',
                 xy=(C_TERPILIH, MARGIN_TERPILIH),
                 xytext=(0.05, 0.16), textcoords='axes fraction',
                 fontsize=10.5, color='#7d5300',
                 bbox=dict(boxstyle='round,pad=0.5', fc='#fdf3dd', ec=WARNA_AKSEN, alpha=0.95),
                 arrowprops=dict(arrowstyle='->', color=WARNA_AKSEN, lw=2))
ax_bawah.annotate(f'C terpilih = {C_TERPILIH}\nrecall = {float(baris_terpilih["recall"]):.4f} '
                  f'(setara recall terbaik {recall_maks_plot:.4f})',
                  xy=(C_TERPILIH, float(baris_terpilih['recall'])),
                  xytext=(0.05, 0.16), textcoords='axes fraction',
                  fontsize=10.5, color='#7d5300',
                  bbox=dict(boxstyle='round,pad=0.5', fc='#fdf3dd', ec=WARNA_AKSEN, alpha=0.95),
                  arrowprops=dict(arrowstyle='->', color=WARNA_AKSEN, lw=2))

ax_atas.legend(loc='upper right', fontsize=9.5, framealpha=0.93)
ax_bawah.legend(loc='lower left', fontsize=9.5, framealpha=0.93, ncol=2)

plt.tight_layout()
simpan_gambar('svm_margin_vs_C')
plt.show()

# Grafik pendamping: jumlah support vector vs C
plt.figure(figsize=(12, 5))
plt.semilogx(ok_m['C'], ok_m['rasio_SV'] * 100, 'o-', lw=2.4, ms=8, color='#8e44ad')
plt.axvline(C_TERPILIH, color=WARNA_AKSEN, ls='-', lw=2.2)
plt.xlabel('Parameter C (skala logaritmik)')
plt.ylabel('Rasio support vector (%)')
plt.title('Proporsi Support Vector terhadap C - C kecil melibatkan lebih banyak sampel '
          'dalam penentuan hyperplane (keputusan lebih kolektif, lebih stabil)')
for cx, ry in zip(ok_m['C'], ok_m['rasio_SV'] * 100):
    plt.text(cx, ry + 1.2, f'{ry:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
simpan_gambar('svm_support_vector_vs_C')
plt.show()

print('Bacaan grafik (dua panel, sumbu-x nilai C yang sama):')
print('  - Panel atas : lebar margin turun tajam saat C naik.')
print('  - Panel bawah: recall mendatar - kenaikan C tidak dibayar dengan sensitivitas.')
print(f'  - Area berarsir: rentang C = [{c_min_layak}, {c_maks_layak}] yang lolos aturan pemilihan')
print(f'    (recall >= {recall_maks_plot:.4f} - {TOLERANSI_RECALL:.2f}). Di dalam rentang itu, titik')
print(f'    dengan margin terlebar adalah C = {C_TERPILIH} (garis vertikal oranye pada kedua panel).')
print('Pembacaan dilakukan secara vertikal pada nilai C yang identik, bukan dari titik potong')
print('antar-kurva, sehingga kesimpulannya tidak bergantung pada pilihan skala sumbu.')

---

# EKSPERIMEN 4 - Visualisasi Hyperplane

Angka pada Eksperimen 3 dilengkapi bukti visual. Tiga sudut pandang ditampilkan:

- **(a)** Bidang dua fitur paling diskriminatif secara klinis: **HbA1c** x **Kadar Glukosa Darah**,
  lengkap dengan garis keputusan ($f(x)=0$), dua garis margin ($f(x)=\pm 1$), dan support vector.
- **(b)** Proyeksi **PCA 2 komponen** dari kelima fitur terstandardisasi - memperlihatkan hyperplane
  pada ruang yang merangkum seluruh fitur, bukan hanya dua.
- **(c)** Perbandingan langsung bentuk batas keputusan **linear vs RBF** pada bidang yang sama.


In [ ]:
# ============================================================
# CELL 14: Eksperimen 4a - Hyperplane pada Bidang HbA1c x Glukosa Darah
# ============================================================
FITUR_PLOT = ['HbA1c_level', 'blood_glucose_level']
LABEL_PLOT = ['HbA1c (terstandardisasi)', 'Kadar Glukosa Darah (terstandardisasi)']

Xp_df, yp = ambil_subsample(X_all, y_all, N_SUBSAMPLE_PLOT)
Xp2 = Xp_df[FITUR_PLOT].values
yp_arr = np.asarray(yp)

sc2 = StandardScaler().fit(Xp2)
Z2 = sc2.transform(Xp2)

# SVC(kernel='linear') dipakai untuk plot karena menyediakan support_vectors_
svc2 = SVC(kernel='linear', C=C_TERPILIH, class_weight='balanced',
           cache_size=500, random_state=RANDOM_STATE).fit(Z2, yp_arr)
w2 = svc2.coef_.ravel(); b2 = float(svc2.intercept_[0])
margin2 = 2.0 / float(np.linalg.norm(w2))

def gambar_kontur_hyperplane(ax, model, Z, y, judul, label_x, label_y,
                             tampilkan_sv=True, isi_wilayah=True):
    """Gambar scatter + kontur decision_function pada level -1, 0, +1."""
    m1, m2 = Z[:, 0].min() - 0.6, Z[:, 0].max() + 0.6
    n1, n2 = Z[:, 1].min() - 0.6, Z[:, 1].max() + 0.6
    xx, yy = np.meshgrid(np.linspace(m1, m2, 320), np.linspace(n1, n2, 320))
    D = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    if isi_wilayah:
        ax.contourf(xx, yy, D, levels=[D.min(), 0, D.max()],
                    colors=['#eaf6ff', '#ffecec'], alpha=0.85)
    ax.scatter(Z[y == 0, 0], Z[y == 0, 1], s=12, c='#3498db', alpha=0.35,
               edgecolors='none', label='Sehat (kelas 0)')
    ax.scatter(Z[y == 1, 0], Z[y == 1, 1], s=16, c='#e74c3c', alpha=0.55,
               edgecolors='none', label='Diabetes (kelas 1)')
    ax.contour(xx, yy, D, levels=[0], colors=['#2c3e50'], linewidths=2.6)
    ax.contour(xx, yy, D, levels=[-1, 1], colors=['#7f8c8d'], linewidths=1.6,
               linestyles='dashed')
    if tampilkan_sv and hasattr(model, 'support_vectors_'):
        sv = model.support_vectors_
        ax.scatter(sv[:, 0], sv[:, 1], s=42, facecolors='none',
                   edgecolors=WARNA_AKSEN, linewidths=0.9, alpha=0.65,
                   label=f'Support vector (n={len(sv):,})')
    ax.set_xlabel(label_x); ax.set_ylabel(label_y); ax.set_title(judul)
    ax.set_xlim(m1, m2); ax.set_ylim(n1, n2)
    return ax

fig, ax = plt.subplots(figsize=(11.5, 8))
gambar_kontur_hyperplane(
    ax, svc2, Z2, yp_arr,
    judul=(f'Eksperimen 4a - Hyperplane SVM Linear (C={C_TERPILIH}) pada Bidang HbA1c x Glukosa\n'
           f'Garis tebal = hyperplane w.x + b = 0   |   garis putus-putus = batas margin f(x) = -1 dan +1'),
    label_x=LABEL_PLOT[0], label_y=LABEL_PLOT[1])
ax.legend(loc='upper left', fontsize=9.5, framealpha=0.92)
ax.text(0.985, 0.03,
        f'||w|| = {np.linalg.norm(w2):.3f}\nmargin = 2/||w|| = {margin2:.3f}\n'
        f'n_SV = {len(svc2.support_vectors_):,} / {len(Z2):,} '
        f'({len(svc2.support_vectors_)/len(Z2)*100:.1f}%)',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=10,
        bbox=dict(boxstyle='round,pad=0.45', fc='white', ec=WARNA_AKSEN, alpha=0.95))
plt.tight_layout()
simpan_gambar('svm_hyperplane_2fitur')
plt.show()

garis('KESIMPULAN EKSPERIMEN 4a')
print(f'Persamaan hyperplane pada bidang 2 fitur (data terstandardisasi):')
print(f'  f(x) = ({w2[0]:+.4f}) * HbA1c + ({w2[1]:+.4f}) * Glukosa + ({b2:+.4f})')
print(f'  ||w|| = {np.linalg.norm(w2):.4f}   ->   lebar margin = {margin2:.4f}')
print(f'  Support vector: {len(svc2.support_vectors_):,} dari {len(Z2):,} sampel '
      f'({len(svc2.support_vectors_)/len(Z2)*100:.1f}%)')
print()
print('Terlihat kedua kelas terpisah oleh pola bertingkat (HbA1c dan glukosa memiliki nilai')
print('ambang klinis), dan pemisah tersebut dapat dijelaskan dengan satu garis lurus - inilah')
print('bukti visual bahwa hyperplane linear sudah memadai untuk struktur data ini.')

In [ ]:
# ============================================================
# CELL 15: Eksperimen 4b - Hyperplane pada Proyeksi PCA 2 Komponen (5 fitur)
# ============================================================
# PCA di sini murni untuk VISUALISASI: seluruh 5 fitur diringkas ke 2 komponen agar
# hyperplane dapat digambar. Model produksi tetap bekerja pada 5 fitur asli.
sc_pca = StandardScaler().fit(Xp_df[SELECTED_FEATURES].values)
Zp = sc_pca.transform(Xp_df[SELECTED_FEATURES].values)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
Zpca = pca.fit_transform(Zp)
var_jelas = pca.explained_variance_ratio_

svc_pca = SVC(kernel='linear', C=C_TERPILIH, class_weight='balanced',
              cache_size=500, random_state=RANDOM_STATE).fit(Zpca, yp_arr)
w_pca = svc_pca.coef_.ravel()
margin_pca = 2.0 / float(np.linalg.norm(w_pca))

fig, ax = plt.subplots(figsize=(11.5, 8))
gambar_kontur_hyperplane(
    ax, svc_pca, Zpca, yp_arr,
    judul=(f'Eksperimen 4b - Hyperplane SVM Linear pada Ruang PCA 2 Komponen (dari 5 fitur)\n'
           f'Total varians yang dijelaskan: {var_jelas.sum()*100:.1f}%'),
    label_x=f'Komponen Utama 1 ({var_jelas[0]*100:.1f}% varians)',
    label_y=f'Komponen Utama 2 ({var_jelas[1]*100:.1f}% varians)')
ax.legend(loc='upper left', fontsize=9.5, framealpha=0.92)
ax.text(0.985, 0.03,
        f'||w|| = {np.linalg.norm(w_pca):.3f}\nmargin = {margin_pca:.3f}\n'
        f'n_SV = {len(svc_pca.support_vectors_):,}',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=10,
        bbox=dict(boxstyle='round,pad=0.45', fc='white', ec=WARNA_AKSEN, alpha=0.95))
plt.tight_layout()
simpan_gambar('svm_hyperplane_pca')
plt.show()

muatan = pd.DataFrame(pca.components_.T, index=FEATURE_LABELS, columns=['PC1', 'PC2']).round(4)
garis('KESIMPULAN EKSPERIMEN 4b')
print(f'Varians dijelaskan: PC1={var_jelas[0]*100:.1f}%, PC2={var_jelas[1]*100:.1f}%, '
      f'total={var_jelas.sum()*100:.1f}%')
print(f'Margin pada ruang PCA: {margin_pca:.4f}  |  n_SV = {len(svc_pca.support_vectors_):,}')
print('Muatan (loading) tiap fitur pada dua komponen utama:')
display(muatan)
print('Meskipun hanya sebagian varians yang tertangkap dua komponen, pemisahan kelas tetap')
print('mengikuti arah lurus - konsisten dengan temuan pada bidang 2 fitur.')

In [ ]:
# ============================================================
# CELL 16: Eksperimen 4c - Perbandingan Visual Batas Keputusan Linear vs RBF
# ============================================================
svc_rbf2 = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced',
               cache_size=500, random_state=RANDOM_STATE).fit(Z2, yp_arr)

fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))
gambar_kontur_hyperplane(
    axes[0], svc2, Z2, yp_arr,
    judul=f'(a) Kernel LINEAR (C={C_TERPILIH}) - batas keputusan berupa garis lurus',
    label_x=LABEL_PLOT[0], label_y=LABEL_PLOT[1])
gambar_kontur_hyperplane(
    axes[1], svc_rbf2, Z2, yp_arr,
    judul='(b) Kernel RBF (C=1.0, gamma=scale) - batas keputusan melengkung',
    label_x=LABEL_PLOT[0], label_y=LABEL_PLOT[1])
axes[0].legend(loc='upper left', fontsize=9)
axes[1].legend(loc='upper left', fontsize=9)

# Perbandingan kuantitatif pada bidang yang sama
skor_lin = svc2.decision_function(Z2)
skor_rbf = svc_rbf2.decision_function(Z2)
auc_lin2 = roc_auc_score(yp_arr, skor_lin)
auc_rbf2 = roc_auc_score(yp_arr, skor_rbf)
axes[0].text(0.985, 0.03, f'AUC (bidang ini) = {auc_lin2:.4f}\nn_SV = {len(svc2.support_vectors_):,}',
             transform=axes[0].transAxes, ha='right', va='bottom', fontsize=10,
             bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='#2c3e50', alpha=0.95))
axes[1].text(0.985, 0.03, f'AUC (bidang ini) = {auc_rbf2:.4f}\nn_SV = {len(svc_rbf2.support_vectors_):,}',
             transform=axes[1].transAxes, ha='right', va='bottom', fontsize=10,
             bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='#2c3e50', alpha=0.95))

plt.suptitle('Eksperimen 4c - Bentuk Batas Keputusan: Linear vs RBF pada Bidang yang Sama',
             fontsize=14, y=1.0)
plt.tight_layout()
simpan_gambar('svm_boundary_linear_vs_rbf')
plt.show()

garis('KESIMPULAN EKSPERIMEN 4c')
print(f'AUC pada bidang 2 fitur : linear={auc_lin2:.4f}  vs  RBF={auc_rbf2:.4f}  '
      f'(selisih {auc_rbf2 - auc_lin2:+.4f})')
print(f'Jumlah support vector   : linear={len(svc2.support_vectors_):,}  vs  '
      f'RBF={len(svc_rbf2.support_vectors_):,}')
print()
print('Kelengkungan yang dihasilkan RBF hanya mengikuti kerapatan lokal titik-titik di sekitar')
print('ambang klinis, bukan menangkap struktur non-linear yang benar-benar baru. Konsekuensinya')
print('bentuk yang lebih rumit itu tidak berubah menjadi keuntungan performa, sementara model')
print('kehilangan sifat yang penting untuk skripsi ini: batas keputusan yang bisa dituliskan')
print('sebagai satu persamaan dan dijelaskan kepada tenaga medis.')

---

# EKSPERIMEN 5 - Interpretasi Vektor Bobot w

Hyperplane linear sepenuhnya ditentukan oleh vektor bobot $w$ dan bias $b$. Karena seluruh fitur
sudah **distandardisasi** (mean 0, simpangan baku 1), besaran $|w_j|$ dapat dibandingkan langsung
antar-fitur: ia menyatakan **seberapa jauh** fungsi keputusan bergeser bila fitur ke-$j$ naik satu
simpangan baku. Tanda $w_j$ menyatakan **arah pengaruh** (positif = menaikkan risiko diabetes).

**Catatan implementasi:** di pipeline produksi, `LinearSVC` dibungkus `CalibratedClassifierCV`,
sehingga `coef_` tidak lagi tersedia di permukaan objek - kalibrasi menyimpan beberapa salinan
estimator hasil cross-validation di `calibrated_classifiers_`. Untuk interpretasi dipakai
**satu `LinearSVC` yang dilatih terpisah pada data scaled + SMOTE yang sama**, karena inilah
hyperplane tunggal yang tidak ambigu (rata-rata beberapa fold justru mengaburkan makna). Bobot dari
estimator di dalam objek kalibrasi tetap dibaca sebagai **pemeriksaan silang** bila API-nya tersedia.

Urutan kepentingan dari $|w|$ juga dibandingkan dengan **permutation importance** (`scoring='recall'`)
yang bersifat model-agnostik, sebagai validasi silang bahwa arah dan urutan pengaruh konsisten.


In [ ]:
# ============================================================
# CELL 17: Eksperimen 5 - Vektor Bobot w dan Permutation Importance
# ============================================================
garis('EKSPERIMEN 5: INTERPRETASI VEKTOR BOBOT HYPERPLANE')
print(f'Data     : {len(Xf_tr):,} latih (DATA PENUH - LinearSVC berbiaya mendekati linear)')
print(f'Model    : LinearSVC(C={C_TERPILIH}) pada data scaled + SMOTE')
print('Estimasi : 1-3 menit')
print('-' * 70)

# (1) Hyperplane tunggal untuk interpretasi
sc_w, Xw_res, yw_res = siapkan_scale_smote(Xf_tr, yf_tr)
lin_w = LinearSVC(C=C_TERPILIH, max_iter=5000, class_weight='balanced',
                  dual=False, random_state=RANDOM_STATE).fit(Xw_res, yw_res)
w_vec = lin_w.coef_.ravel()
b_val = float(lin_w.intercept_[0])
norma_w_final = float(np.linalg.norm(w_vec))
margin_final  = 2.0 / norma_w_final
print(f'||w|| = {norma_w_final:.4f}   margin = 2/||w|| = {margin_final:.4f}   b = {b_val:+.4f}')

# (2) Pemeriksaan silang: bobot dari estimator di dalam CalibratedClassifierCV
def ambil_coef_dari_kalibrasi(pipe):
    """CalibratedClassifierCV menyimpan estimator hasil tiap fold. Nama atributnya
    berbeda antar versi scikit-learn ('estimator' >= 1.2, 'base_estimator' < 1.2),
    sehingga diakses defensif. Bila gagal, interpretasi tetap memakai lin_w."""
    try:
        cal = pipe.named_steps['clf']
        kumpulan = []
        for sub in cal.calibrated_classifiers_:
            for atr in ('estimator', 'base_estimator'):
                est = getattr(sub, atr, None)
                if est is not None and hasattr(est, 'coef_'):
                    kumpulan.append(est.coef_.ravel())
                    break
        if kumpulan:
            return np.mean(np.vstack(kumpulan), axis=0)
    except Exception as e:
        print(f'  (info) bobot dari objek kalibrasi tidak dapat dibaca: {type(e).__name__}')
    return None

pipe_kal = buat_pipeline_svm(kernel='linear', C=C_TERPILIH, max_iter=5000)
pipe_kal.fit(Xf_tr, yf_tr)
w_kal = ambil_coef_dari_kalibrasi(pipe_kal)
if w_kal is not None:
    korelasi_w = float(np.corrcoef(w_vec, w_kal)[0, 1])
    print(f'Pemeriksaan silang: korelasi bobot LinearSVC terpisah vs rata-rata estimator '
          f'di dalam kalibrasi = {korelasi_w:.4f}')
else:
    korelasi_w = None
    print('Pemeriksaan silang bobot kalibrasi dilewati (API versi scikit-learn berbeda).')

# (3) Tabel bobot
abs_w = np.abs(w_vec)
tabel_bobot_w = pd.DataFrame({
    'fitur'            : FEATURE_LABELS,
    'nama_kolom'       : SELECTED_FEATURES,
    'koefisien_w'      : w_vec,
    'abs_w'            : abs_w,
    'abs_w_ternormalisasi': abs_w / abs_w.sum(),
    'arah_pengaruh'    : ['Menaikkan risiko diabetes' if v > 0 else 'Menurunkan risiko diabetes'
                          for v in w_vec],
}).sort_values('abs_w', ascending=False).reset_index(drop=True)
tabel_bobot_w['peringkat_w'] = np.arange(1, len(tabel_bobot_w) + 1)

# (4) Permutation importance (model-agnostik, scoring='recall')
print(f'\nMenghitung permutation importance pada {N_SUBSAMPLE_PERM:,} sampel uji ...')
Xperm, yperm = ambil_subsample(Xf_te, yf_te, N_SUBSAMPLE_PERM)
pipe_perm = pipeline_linear_mentah(C_TERPILIH).fit(Xf_tr, yf_tr)
perm = permutation_importance(pipe_perm, Xperm, yperm, scoring='recall',
                              n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
tabel_perm = pd.DataFrame({
    'nama_kolom': SELECTED_FEATURES,
    'perm_mean' : perm.importances_mean,
    'perm_std'  : perm.importances_std,
}).sort_values('perm_mean', ascending=False).reset_index(drop=True)
tabel_perm['peringkat_perm'] = np.arange(1, len(tabel_perm) + 1)

tabel_bobot_w = tabel_bobot_w.merge(tabel_perm, on='nama_kolom', how='left')
tabel_bobot_w['selisih_peringkat'] = (tabel_bobot_w['peringkat_w'] -
                                      tabel_bobot_w['peringkat_perm']).abs()
tabel_bobot_w = tabel_bobot_w.round(6)
simpan_tabel(tabel_bobot_w, 'tabel_bobot_w')

# (5) Grafik barh koefisien + perbandingan peringkat
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
urut = tabel_bobot_w.sort_values('koefisien_w')
warna_bar = ['#e74c3c' if v > 0 else '#3498db' for v in urut['koefisien_w']]
axes[0].barh(urut['fitur'], urut['koefisien_w'], color=warna_bar, edgecolor='white')
axes[0].axvline(0, color='#2c3e50', lw=1.2)
axes[0].set_xlabel('Koefisien w (pada fitur terstandardisasi)')
axes[0].set_title('(a) Vektor Bobot Hyperplane\nmerah = menaikkan risiko, biru = menurunkan risiko')
for i, (v, f_) in enumerate(zip(urut['koefisien_w'], urut['fitur'])):
    axes[0].text(v + (0.02 if v >= 0 else -0.02), i, f'{v:+.4f}',
                 va='center', ha='left' if v >= 0 else 'right', fontsize=9.5)

urut_p = tabel_bobot_w.sort_values('perm_mean')
axes[1].barh(urut_p['fitur'], urut_p['perm_mean'],
             xerr=urut_p['perm_std'], color='#8e44ad', edgecolor='white',
             error_kw=dict(ecolor='#5b2c6f', lw=1.2, capsize=4))
axes[1].set_xlabel('Penurunan recall saat fitur diacak')
axes[1].set_title('(b) Permutation Importance (scoring = recall)\nvalidasi silang model-agnostik')

plt.suptitle('Eksperimen 5 - Interpretasi Hyperplane: Bobot w dan Kepentingan Fitur',
             fontsize=14, y=1.02)
plt.tight_layout()
simpan_gambar('svm_bobot_w')
plt.show()

garis('KESIMPULAN EKSPERIMEN 5')
print('Persamaan hyperplane pada 5 fitur terstandardisasi:')
suku = '  f(x) = ' + ' '.join(
    [f'({w_vec[i]:+.4f})*{SELECTED_FEATURES[i]}' for i in range(len(SELECTED_FEATURES))]) + f' ({b_val:+.4f})'
print(suku)
print(f'  Prediksi positif bila f(x) >= threshold operasional.')
print()
print('Urutan kepentingan menurut |w| vs permutation importance:')
display(tabel_bobot_w[['fitur', 'koefisien_w', 'abs_w_ternormalisasi', 'peringkat_w',
                       'perm_mean', 'peringkat_perm', 'selisih_peringkat', 'arah_pengaruh']])
n_sama = int((tabel_bobot_w['selisih_peringkat'] == 0).sum())
print(f'Peringkat identik pada {n_sama} dari {len(tabel_bobot_w)} fitur; '
      f'rata-rata pergeseran peringkat = {tabel_bobot_w["selisih_peringkat"].mean():.2f}.')
print('Kesesuaian ini menunjukkan bobot hyperplane bukan artefak numerik, melainkan benar-benar')
print('mencerminkan kontribusi tiap fitur - sekaligus alasan tambahan memilih kernel linear:')
print('hyperplane-nya dapat dibaca sebagai pernyataan klinis yang jelas.')

---

# EKSPERIMEN 6 - Uji Signifikansi Statistik: Linear vs RBF

Perbedaan angka antar-kernel pada Eksperimen 1 dan 2 belum tentu berarti secara statistik. Karena
itu kedua kernel dievaluasi dengan **StratifiedKFold 5-fold** pada data yang sama (fold yang sama
persis untuk kedua model, sehingga sampelnya **berpasangan**), lalu diuji dengan:

- **Paired t-test** (`scipy.stats.ttest_rel`) - uji parametrik untuk selisih rata-rata.
- **Wilcoxon signed-rank** (`scipy.stats.wilcoxon`) - versi non-parametrik, tidak mengasumsikan
  normalitas (penting karena n fold hanya 5).

Bila **tidak** ada perbedaan signifikan (p >= 0,05), maka pemilihan kernel linear dijustifikasi
dengan prinsip **parsimony (Occam's razor)**: di antara dua model yang performanya tidak dapat
dibedakan secara statistik, dipilih model yang lebih sederhana, lebih cepat, dan lebih dapat
dijelaskan.


In [ ]:
# ============================================================
# CELL 18: Eksperimen 6 - Uji Signifikansi Linear vs RBF (5-fold berpasangan)
# ============================================================
garis('EKSPERIMEN 6: UJI SIGNIFIKANSI LINEAR VS RBF')
print(f'Data     : {len(X_sv):,} sampel (subsample stratified, dibatasi biaya SVC-RBF)')
print('Skema    : StratifiedKFold(5) - fold identik untuk kedua kernel (berpasangan)')
print('Estimasi : 2-5 menit')
print('-' * 70)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X_uji = X_sv.reset_index(drop=True)
y_uji = y_sv.reset_index(drop=True)

skor_fold = {'linear': {'auc': [], 'recall': [], 'f1': [], 'waktu': []},
             'rbf'   : {'auc': [], 'recall': [], 'f1': [], 'waktu': []}}

for f_i, (idx_tr, idx_te) in enumerate(skf.split(X_uji, y_uji), 1):
    Xtr, Xte = X_uji.iloc[idx_tr], X_uji.iloc[idx_te]
    ytr, yte = y_uji.iloc[idx_tr], y_uji.iloc[idx_te]
    pesan = f'Fold {f_i}/5: '
    for nama, pipa in [('linear', pipeline_linear_mentah(C_TERPILIH)),
                       ('rbf',    pipeline_svc_mentah('rbf', C=1.0, gamma='scale'))]:
        try:
            r = evaluasi_decision(pipa, Xtr, ytr, Xte, yte)
            skor_fold[nama]['auc'].append(r['roc_auc'])
            skor_fold[nama]['recall'].append(r['recall'])
            skor_fold[nama]['f1'].append(r['f1'])
            skor_fold[nama]['waktu'].append(r['waktu_latih_s'])
            pesan += f'{nama} AUC={r["roc_auc"]:.4f} ({r["waktu_latih_s"]:.1f}s)  '
        except Exception as e:
            skor_fold[nama]['auc'].append(np.nan)
            skor_fold[nama]['recall'].append(np.nan)
            skor_fold[nama]['f1'].append(np.nan)
            skor_fold[nama]['waktu'].append(np.nan)
            pesan += f'{nama} GAGAL({type(e).__name__})  '
    print(pesan)

def uji_berpasangan(a, b):
    """Paired t-test + Wilcoxon signed-rank. Wilcoxon bisa gagal bila seluruh
    selisih nol atau n terlalu kecil, sehingga dibungkus try/except."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    valid = ~(np.isnan(a) | np.isnan(b))
    a, b = a[valid], b[valid]
    hasil = {'n_fold': int(len(a)),
             'mean_linear': float(np.mean(a)) if len(a) else np.nan,
             'mean_rbf'   : float(np.mean(b)) if len(b) else np.nan,
             'std_linear' : float(np.std(a, ddof=1)) if len(a) > 1 else np.nan,
             'std_rbf'    : float(np.std(b, ddof=1)) if len(b) > 1 else np.nan}
    hasil['selisih_linear_minus_rbf'] = hasil['mean_linear'] - hasil['mean_rbf']
    try:
        t_stat, p_t = stats.ttest_rel(a, b)
        hasil['t_stat'], hasil['p_ttest'] = float(t_stat), float(p_t)
    except Exception:
        hasil['t_stat'], hasil['p_ttest'] = np.nan, np.nan
    try:
        w_stat, p_w = stats.wilcoxon(a, b, zero_method='zsplit')
        hasil['w_stat'], hasil['p_wilcoxon'] = float(w_stat), float(p_w)
    except Exception:
        hasil['w_stat'], hasil['p_wilcoxon'] = np.nan, np.nan
    hasil['signifikan_alpha_0.05'] = bool(hasil['p_ttest'] < 0.05) if not np.isnan(hasil['p_ttest']) else False
    return hasil

baris_uji = []
for metrik in ['auc', 'recall', 'f1']:
    h = uji_berpasangan(skor_fold['linear'][metrik], skor_fold['rbf'][metrik])
    h = {'metrik': metrik.upper(), **h}
    baris_uji.append(h)

tabel_uji_linear_vs_rbf = pd.DataFrame(baris_uji).round(6)
simpan_tabel(tabel_uji_linear_vs_rbf, 'tabel_uji_linear_vs_rbf')

# Tabel skor per fold (transparansi data mentah uji)
tabel_fold = pd.DataFrame({
    'fold'          : np.arange(1, 6),
    'auc_linear'    : skor_fold['linear']['auc'],
    'auc_rbf'       : skor_fold['rbf']['auc'],
    'recall_linear' : skor_fold['linear']['recall'],
    'recall_rbf'    : skor_fold['rbf']['recall'],
    'waktu_linear_s': skor_fold['linear']['waktu'],
    'waktu_rbf_s'   : skor_fold['rbf']['waktu'],
}).round(6)
simpan_tabel(tabel_fold, 'tabel_fold_linear_vs_rbf')

baris_auc = tabel_uji_linear_vs_rbf[tabel_uji_linear_vs_rbf['metrik'] == 'AUC'].iloc[0]
selisih_auc = float(baris_auc['selisih_linear_minus_rbf'])
p_ttest_auc = float(baris_auc['p_ttest'])
p_wil_auc   = float(baris_auc['p_wilcoxon'])
waktu_lin   = float(np.nanmean(skor_fold['linear']['waktu']))
waktu_rbf   = float(np.nanmean(skor_fold['rbf']['waktu']))
rasio_waktu = waktu_rbf / max(waktu_lin, 1e-9)
SIGNIFIKAN_RBF = bool(p_ttest_auc < 0.05) if not np.isnan(p_ttest_auc) else False

garis('KESIMPULAN EKSPERIMEN 6')
print(f'AUC rata-rata linear : {baris_auc["mean_linear"]:.4f} (SD {baris_auc["std_linear"]:.4f})')
print(f'AUC rata-rata RBF    : {baris_auc["mean_rbf"]:.4f} (SD {baris_auc["std_rbf"]:.4f})')
print(f'Selisih (linear-RBF) : {selisih_auc:+.4f}')
print(f'Paired t-test        : t={baris_auc["t_stat"]:.4f}, p={p_ttest_auc:.4f}')
print(f'Wilcoxon signed-rank : W={baris_auc["w_stat"]:.4f}, p={p_wil_auc:.4f}')
print(f'Waktu latih rata-rata: linear={waktu_lin:.2f}s vs RBF={waktu_rbf:.2f}s '
      f'({rasio_waktu:.1f}x lebih lambat)')
print()
if SIGNIFIKAN_RBF and selisih_auc < 0:
    print('Hasil: RBF unggul secara SIGNIFIKAN. Namun keunggulan itu perlu ditimbang terhadap')
    print(f'biaya komputasi {rasio_waktu:.1f}x dan hilangnya interpretabilitas hyperplane.')
else:
    print('Hasil: TIDAK ADA perbedaan signifikan pada alpha = 0,05.')
    print('Justifikasi memilih hyperplane LINEAR (prinsip parsimony / Occam s razor):')
    print(f'  1. Performa: selisih AUC hanya {abs(selisih_auc):.4f} dan p = {p_ttest_auc:.4f} '
          f'(>= 0,05) -> tidak dapat dibedakan secara statistik.')
    print(f'  2. Kecepatan: pelatihan {rasio_waktu:.1f}x lebih cepat, dan inferensi linear hanya')
    print('     memerlukan satu perkalian titik w.x + b (bukan evaluasi kernel terhadap ribuan SV).')
    print('  3. Skalabilitas: LinearSVC dapat dilatih pada seluruh 96.146 baris, sedangkan')
    print('     SVC-RBF terpaksa memakai subsample.')
    print('  4. Interpretabilitas: bobot w dapat dibaca sebagai kontribusi klinis tiap fitur')
    print('     (Eksperimen 5) - syarat penting untuk sistem pendukung keputusan medis.')

---

# EKSPERIMEN 7 - Kalibrasi Probabilitas

SVM secara alami menghasilkan **skor jarak** ke hyperplane (`decision_function`), bukan probabilitas.
Padahal sistem DiaPredict mengambil keputusan berdasarkan **ambang probabilitas** dan menampilkan
angka risiko kepada pengguna. Karena itu keluaran SVM **wajib dikalibrasi**, dan pilihan metode
kalibrasi perlu dibuktikan - bukan sekadar diwarisi dari notebook sebelumnya.

Tiga varian dibandingkan dengan **Brier score** (semakin kecil semakin baik; mengukur ketepatan
probabilitas, bukan sekadar urutan) dan **reliability diagram**:

1. `CalibratedClassifierCV(method='sigmoid')` - Platt scaling (dipakai V2).
2. `CalibratedClassifierCV(method='isotonic')` - regresi isotonik, lebih fleksibel tetapi lebih
   rawan overfitting pada data kecil.
3. **Tanpa kalibrasi** - `decision_function` yang dinormalisasi min-max ke rentang [0,1] sebagai
   pengganti probabilitas (praktik yang sering dilakukan tetapi tidak memiliki dasar probabilistik).


In [ ]:
# ============================================================
# CELL 19: Eksperimen 7 - Perbandingan Metode Kalibrasi Probabilitas
# ============================================================
garis('EKSPERIMEN 7: KALIBRASI PROBABILITAS SVM')
print(f'Data     : {len(Xk_tr):,} latih / {len(Xk_te):,} uji')
print(f'Model dasar: LinearSVC(C={C_TERPILIH})')
print('Estimasi : 1-2 menit')
print('-' * 70)

hasil_kalibrasi = []
kurva_kalibrasi = {}

# (1) & (2) Kalibrasi sigmoid dan isotonic
for metode in ['sigmoid', 'isotonic']:
    try:
        t0 = time.time()
        pipa = buat_pipeline_svm(kernel='linear', C=C_TERPILIH, max_iter=5000, kalibrasi=metode)
        pipa.fit(Xk_tr, yk_tr)
        proba = pipa.predict_proba(Xk_te)[:, 1]
        thr = threshold_youden(yk_te, proba)
        met = hitung_metrik(yk_te, (proba >= thr).astype(int), proba)
        frac_pos, mean_pred = calibration_curve(yk_te, proba, n_bins=10, strategy='quantile')
        kurva_kalibrasi[metode] = (mean_pred, frac_pos)
        hasil_kalibrasi.append({
            'metode': f'CalibratedClassifierCV ({metode})', 'brier': met['brier'],
            'roc_auc': met['roc_auc'], 'recall': met['recall'], 'precision': met['precision'],
            'f1': met['f1'], 'threshold_youden': thr, 'waktu_latih_s': time.time() - t0,
            'probabilitas_valid': 'Ya', 'status': 'OK'})
        print(f'{metode:<10} Brier={met["brier"]:.5f}  AUC={met["roc_auc"]:.4f}  '
              f'recall={met["recall"]:.4f}  ({time.time()-t0:.1f}s)')
    except Exception as e:
        hasil_kalibrasi.append({'metode': f'CalibratedClassifierCV ({metode})', 'brier': np.nan,
                                'roc_auc': np.nan, 'recall': np.nan, 'precision': np.nan,
                                'f1': np.nan, 'threshold_youden': np.nan, 'waktu_latih_s': np.nan,
                                'probabilitas_valid': 'Ya', 'status': f'GAGAL: {type(e).__name__}'})
        print(f'{metode:<10} GAGAL ({type(e).__name__}) -> dilewati')

# (3) Tanpa kalibrasi: decision_function dinormalisasi min-max ke [0,1]
try:
    t0 = time.time()
    pipa_mentah = pipeline_linear_mentah(C_TERPILIH).fit(Xk_tr, yk_tr)
    skor_mentah = pipa_mentah.decision_function(Xk_te)
    proba_semu = (skor_mentah - skor_mentah.min()) / (skor_mentah.max() - skor_mentah.min() + 1e-12)
    thr = threshold_youden(yk_te, proba_semu)
    met = hitung_metrik(yk_te, (proba_semu >= thr).astype(int), proba_semu)
    frac_pos, mean_pred = calibration_curve(yk_te, proba_semu, n_bins=10, strategy='quantile')
    kurva_kalibrasi['tanpa kalibrasi'] = (mean_pred, frac_pos)
    hasil_kalibrasi.append({
        'metode': 'Tanpa kalibrasi (decision_function min-max)', 'brier': met['brier'],
        'roc_auc': met['roc_auc'], 'recall': met['recall'], 'precision': met['precision'],
        'f1': met['f1'], 'threshold_youden': thr, 'waktu_latih_s': time.time() - t0,
        'probabilitas_valid': 'Tidak', 'status': 'OK'})
    print(f'{"tanpa":<10} Brier={met["brier"]:.5f}  AUC={met["roc_auc"]:.4f}  '
          f'recall={met["recall"]:.4f}  ({time.time()-t0:.1f}s)')
except Exception as e:
    print(f'Varian tanpa kalibrasi GAGAL ({type(e).__name__}) -> dilewati')

tabel_kalibrasi_svm = pd.DataFrame(hasil_kalibrasi).round(6)
simpan_tabel(tabel_kalibrasi_svm, 'tabel_kalibrasi_svm')

# --- Reliability diagram + distribusi probabilitas ---
fig, axes = plt.subplots(1, 2, figsize=(17, 6.5))
warna_kal = {'sigmoid': WARNA_MODEL['SVM (Linear)'], 'isotonic': '#9b59b6',
             'tanpa kalibrasi': '#e74c3c'}
penanda = {'sigmoid': 'o', 'isotonic': 's', 'tanpa kalibrasi': '^'}

axes[0].plot([0, 1], [0, 1], 'k:', lw=1.8, label='Kalibrasi sempurna')
for nama, (mp, fp) in kurva_kalibrasi.items():
    axes[0].plot(mp, fp, marker=penanda[nama], lw=2.2, ms=7,
                 color=warna_kal[nama], label=nama)
axes[0].set_xlabel('Probabilitas rata-rata yang diprediksi')
axes[0].set_ylabel('Proporsi kasus positif sebenarnya')
axes[0].set_title('(a) Reliability Diagram - semakin dekat garis diagonal semakin baik')
axes[0].legend(loc='upper left', fontsize=10)
axes[0].set_xlim(-0.02, 1.02); axes[0].set_ylim(-0.02, 1.02)

ok_kal = tabel_kalibrasi_svm[tabel_kalibrasi_svm['status'] == 'OK']
warna_brier = [WARNA_MODEL['SVM (Linear)'] if 'sigmoid' in m else
               ('#9b59b6' if 'isotonic' in m else '#e74c3c') for m in ok_kal['metode']]
axes[1].bar(range(len(ok_kal)), ok_kal['brier'], color=warna_brier, edgecolor='white')
label_kal = ['sigmoid\n(Platt)' if 'sigmoid' in m else
             ('isotonic' if 'isotonic' in m else 'tanpa\nkalibrasi') for m in ok_kal['metode']]
axes[1].set_xticks(range(len(ok_kal)))
axes[1].set_xticklabels(label_kal, fontsize=10)
axes[1].set_ylabel('Brier score (semakin kecil semakin baik)')
axes[1].set_title('(b) Brier Score per Metode Kalibrasi')
for i, v in enumerate(ok_kal['brier']):
    axes[1].text(i, v * 1.02, f'{v:.5f}', ha='center', fontsize=10)

plt.suptitle('Eksperimen 7 - Justifikasi Kalibrasi Probabilitas SVM', fontsize=14, y=1.02)
plt.tight_layout()
simpan_gambar('svm_kalibrasi')
plt.show()

garis('KESIMPULAN EKSPERIMEN 7')
if len(ok_kal):
    terbaik_kal = ok_kal.loc[ok_kal['brier'].idxmin()]
    METODE_KALIBRASI_TERPILIH = str(terbaik_kal['metode'])
    BRIER_TERPILIH = float(terbaik_kal['brier'])
    print(f'Brier terendah : {METODE_KALIBRASI_TERPILIH} (Brier={BRIER_TERPILIH:.5f})')
    baris_tanpa = ok_kal[ok_kal['probabilitas_valid'] == 'Tidak']
    if len(baris_tanpa):
        b_tanpa = float(baris_tanpa.iloc[0]['brier'])
        print(f'Tanpa kalibrasi: Brier={b_tanpa:.5f} '
              f'({(b_tanpa - BRIER_TERPILIH) / max(BRIER_TERPILIH, 1e-12) * 100:+.1f}% lebih buruk)')
    print(f'ROC-AUC seluruh varian praktis sama (rentang '
          f'{ok_kal["roc_auc"].max() - ok_kal["roc_auc"].min():.5f}) - ini wajar karena kalibrasi')
    print('adalah transformasi monoton: ia memperbaiki NILAI probabilitas, bukan URUTAN peringkat.')
    print()
    print('Implikasi untuk sistem: karena DiaPredict menampilkan angka risiko dan memakai ambang')
    print('probabilitas, kalibrasi bukan pelengkap melainkan syarat agar angka yang ditampilkan')
    print('benar-benar bermakna (mis. "risiko 70%" memang terjadi pada sekitar 70% kasus serupa).')
else:
    METODE_KALIBRASI_TERPILIH, BRIER_TERPILIH = 'sigmoid', float('nan')
    print('Seluruh varian kalibrasi gagal - lihat kolom status pada tabel.')

---

# KESIMPULAN - Argumen Terstruktur Pemilihan Hyperplane

Cell berikut merangkum seluruh bukti menjadi satu argumen berurutan dan menyimpannya ke
`hasil_svm_hyperplane.json` (dibaca notebook `06` dan website).


In [ ]:
# ============================================================
# CELL 20: Kesimpulan Terstruktur + Penyimpanan JSON hasil_svm_hyperplane
# ============================================================
ok_kern = tabel_perbandingan_kernel[tabel_perbandingan_kernel['status'] == 'OK'].copy()

def ambil_baris_kernel(nama_kernel, kolom_urut='roc_auc'):
    sub = ok_kern[ok_kern['kernel'] == nama_kernel]
    if len(sub) == 0:
        return None
    return sub.sort_values(kolom_urut, ascending=False).iloc[0]

b_lin  = ambil_baris_kernel('linear')
b_rbf  = ambil_baris_kernel('rbf')
b_poly = ok_kern[ok_kern['kernel'] == 'poly'].sort_values('roc_auc', ascending=False)
b_poly = b_poly.iloc[0] if len(b_poly) else None
b_sig  = ambil_baris_kernel('sigmoid')

def frasa_penolakan(baris, nama):
    if baris is None or b_lin is None:
        return f'- {nama}: tidak berhasil dilatih pada anggaran komputasi yang tersedia (lihat kolom status).'
    d_auc = baris['roc_auc'] - b_lin['roc_auc']
    d_rec = baris['recall'] - b_lin['recall']
    rasio = baris['waktu_latih_s'] / max(b_lin['waktu_latih_s'], 1e-9)
    return (f'- {nama}: AUC {baris["roc_auc"]:.4f} ({d_auc:+.4f} terhadap linear), '
            f'recall {baris["recall"]:.4f} ({d_rec:+.4f}), '
            f'waktu latih {baris["waktu_latih_s"]:.1f}s ({rasio:.1f}x kernel linear).')

garis('ARGUMEN TERSTRUKTUR: KENAPA HYPERPLANE INI YANG DIPILIH')
print('[1] KERNEL TERPILIH: LINEAR')
print(f'    Hyperplane dicari di ruang fitur asli (5 fitur terstandardisasi), bukan di ruang')
print(f'    berdimensi lebih tinggi. Bukti dari Eksperimen 1, 2, 4, dan 6:')
print(f'    - Kernel linear terbaik : AUC={b_lin["roc_auc"]:.4f}, recall={b_lin["recall"]:.4f}, '
      f'latih={b_lin["waktu_latih_s"]:.1f}s')
print(f'    - Uji 5-fold berpasangan: selisih AUC linear-RBF = {selisih_auc:+.4f}, '
      f'p(t-test)={p_ttest_auc:.4f}, p(Wilcoxon)={p_wil_auc:.4f}')
print(f'    - Kesimpulan uji        : '
      f'{"perbedaan SIGNIFIKAN" if SIGNIFIKAN_RBF else "TIDAK signifikan pada alpha=0,05"}')
print()
print('[2] ALASAN MENOLAK KERNEL LAIN (dengan angka):')
print(frasa_penolakan(b_rbf,  'RBF'))
print(frasa_penolakan(b_poly, 'Polinomial'))
print(frasa_penolakan(b_sig,  'Sigmoid'))
print(f'    Tambahan: SVC berkernel berkompleksitas O(n^2)-O(n^3) sehingga hanya dapat dilatih')
print(f'    pada subsample {len(Xk_tr):,} baris, sedangkan LinearSVC dilatih pada seluruh '
      f'{len(Xf_tr):,} baris.')
print()
print(f'[3] NILAI C TERPILIH: {C_TERPILIH}')
print(f'    Aturan pemilihan ditetapkan di muka: margin TERLEBAR di antara kandidat yang recall-nya')
print(f'    masih dalam {TOLERANSI_RECALL*100:.0f} poin persen dari recall terbaik ({recall_maks:.4f}).')
print(f'    - ||w||                 : {NORMA_W_TERPILIH:.4f}')
print(f'    - Lebar margin 2/||w||  : {MARGIN_TERPILIH:.4f}')
print(f'    - Jumlah support vector : {NSV_TERPILIH:,} ({RASIO_SV_TERPILIH*100:.1f}% dari sampel latih)')
print(f'    - Recall / Precision    : {float(baris_terpilih["recall"]):.4f} / '
      f'{float(baris_terpilih["precision"]):.4f}')
print(f'    - Konfigurasi V2 (C={PARAM_SVM_V2["C"]}) : '
      f'{"TERKONFIRMASI oleh eksperimen ini" if abs(C_TERPILIH - PARAM_SVM_V2["C"]) < 1e-9 else "diperbarui berdasarkan eksperimen ini"}')
print()
print('[4] HYPERPLANE FINAL PADA DATA PENUH (5 fitur terstandardisasi):')
print('    f(x) = ' + ' '.join([f'({w_vec[i]:+.4f})*{SELECTED_FEATURES[i]}'
                                for i in range(len(SELECTED_FEATURES))]) + f' ({b_val:+.4f})')
print(f'    ||w|| = {norma_w_final:.4f}  ->  lebar margin = {margin_final:.4f}')
print(f'    Fitur paling menentukan: {tabel_bobot_w.iloc[0]["fitur"]} '
      f'(|w| ternormalisasi = {float(tabel_bobot_w.iloc[0]["abs_w_ternormalisasi"])*100:.1f}%)')
print()
print(f'[5] KALIBRASI: {METODE_KALIBRASI_TERPILIH} (Brier={BRIER_TERPILIH:.5f})')
print('    Diperlukan karena sistem memakai ambang probabilitas, sedangkan SVM hanya menghasilkan')
print('    skor jarak ke hyperplane.')
print()
print('[6] INTI JAWABAN UNTUK PENGUJI:')
print('    Hyperplane yang dipilih adalah hyperplane LINEAR dengan margin TERLEBAR yang masih')
print('    memenuhi kebutuhan sensitivitas skrining medis. Bukan hyperplane sembarang, bukan pula')
print('    hyperplane paling ketat: C yang lebih besar hanya mempersempit margin tanpa menaikkan')
print('    recall, sedangkan kernel non-linear hanya menambah biaya komputasi tanpa keunggulan')
print('    performa yang signifikan secara statistik.')

# ---------------- Penyimpanan JSON (kontrak spec) ----------------
kesimpulan_teks = (
    f'Hyperplane SVM yang dipilih adalah hyperplane linear dengan C={C_TERPILIH}, '
    f'menghasilkan norma bobot ||w||={NORMA_W_TERPILIH:.4f}, lebar margin 2/||w||={MARGIN_TERPILIH:.4f}, '
    f'dan {NSV_TERPILIH:,} support vector ({RASIO_SV_TERPILIH*100:.1f}% dari sampel latih). '
    f'Pemilihan didasarkan pada aturan margin terlebar yang masih menjaga recall dalam '
    f'{TOLERANSI_RECALL*100:.0f} poin persen dari recall terbaik ({recall_maks:.4f}). '
    f'Kernel non-linear ditolak karena selisih AUC terhadap kernel linear hanya '
    f'{selisih_auc:+.4f} dengan p(t-test)={p_ttest_auc:.4f} dan p(Wilcoxon)={p_wil_auc:.4f} '
    f'(tidak signifikan pada alpha=0,05), sementara biaya pelatihannya {rasio_waktu:.1f} kali lipat. '
    f'Sesuai prinsip parsimony, dipilih model paling sederhana yang performanya tidak dapat '
    f'dibedakan secara statistik, sekaligus paling cepat, paling skalabel, dan paling dapat '
    f'diinterpretasi secara klinis.'
)

hasil_svm_hyperplane = {
    'kernel_terpilih': 'linear',
    'C_terpilih': C_TERPILIH,
    'implementasi': 'LinearSVC (formulasi primal, dual=False) + CalibratedClassifierCV(cv=3)',
    'aturan_pemilihan': (f'Margin terlebar di antara kandidat dengan recall >= recall_maks - '
                         f'{TOLERANSI_RECALL} (recall_maks={recall_maks:.4f})'),
    'perbandingan_kernel': tabel_perbandingan_kernel.to_dict('records'),
    'grid_C_gamma': {
        'linear': tabel_grid_C_linear.to_dict('records'),
        'rbf'   : tabel_rbf.to_dict('records'),
        'gabungan': tabel_grid_C_gamma.to_dict('records'),
    },
    'analisis_margin': tabel_analisis_margin.to_dict('records'),
    'hyperplane_final': {
        'fitur'        : SELECTED_FEATURES,
        'koefisien_w'  : [float(v) for v in w_vec],
        'intercept_b'  : b_val,
        'norma_w'      : norma_w_final,
        'lebar_margin' : margin_final,
        'n_support_vector': NSV_TERPILIH,
        'rasio_support_vector': RASIO_SV_TERPILIH,
        'catatan': ('norma_w dan lebar_margin dihitung dari LinearSVC pada data penuh; '
                    'n_support_vector dihitung dengan SVC(kernel="linear") pada subsample '
                    'karena LinearSVC (primal) tidak menyimpan support_vectors_'),
    },
    'bobot_w': {
        'tabel': tabel_bobot_w.to_dict('records'),
        'peta_fitur_koefisien': {k: float(v) for k, v in zip(SELECTED_FEATURES, w_vec)},
        'korelasi_dengan_estimator_kalibrasi': korelasi_w,
    },
    'uji_linear_vs_rbf': {
        'ringkasan_uji': tabel_uji_linear_vs_rbf.to_dict('records'),
        'skor_per_fold': tabel_fold.to_dict('records'),
        'selisih_auc_linear_minus_rbf': selisih_auc,
        'p_ttest_auc': p_ttest_auc,
        'p_wilcoxon_auc': p_wil_auc,
        'signifikan_alpha_0.05': SIGNIFIKAN_RBF,
        'rasio_waktu_rbf_per_linear': rasio_waktu,
    },
    'kalibrasi': {
        'metode_terpilih': METODE_KALIBRASI_TERPILIH,
        'brier_terpilih' : BRIER_TERPILIH,
        'tabel'          : tabel_kalibrasi_svm.to_dict('records'),
    },
    'keterbatasan': (f'Perbandingan kernel dijalankan pada subsample stratified '
                     f'{N_SUBSAMPLE_KERNEL:,} baris (grid RBF {N_SUBSAMPLE_SV:,} baris) karena '
                     f'SVC berkernel berkompleksitas O(n^2)-O(n^3). Kesimpulan bersifat komparatif '
                     f'antar-konfigurasi, bukan estimasi performa absolut pada data penuh.'),
    'kesimpulan': kesimpulan_teks,
}

simpan_json(hasil_svm_hyperplane, 'hasil_svm_hyperplane')

garis('SELESAI - NOTEBOOK 03')
print(f'Tabel  : {OUTPUT_DIR}/tabel/')
print(f'Gambar : {OUTPUT_DIR}/gambar/')
print(f'JSON   : {OUTPUT_DIR}/json/hasil_svm_hyperplane.json')
print()
print('Gambar yang dihasilkan:')
for g in ['svm_perbandingan_kernel', 'svm_kurva_C', 'svm_heatmap_rbf', 'svm_margin_vs_C',
          'svm_support_vector_vs_C', 'svm_hyperplane_2fitur', 'svm_hyperplane_pca',
          'svm_boundary_linear_vs_rbf', 'svm_bobot_w', 'svm_kalibrasi']:
    print(f'  - {g}.png')
print()
print('Tabel yang dihasilkan:')
for t in ['tabel_perbandingan_kernel', 'tabel_grid_C_linear', 'tabel_grid_C_gamma',
          'tabel_analisis_margin', 'tabel_bobot_w', 'tabel_uji_linear_vs_rbf',
          'tabel_fold_linear_vs_rbf', 'tabel_kalibrasi_svm']:
    print(f'  - {t}.csv')

---

# RINGKASAN UNTUK SKRIPSI

> Paragraf berikut siap disalin ke Bab IV / pembahasan sebagai jawaban atas pertanyaan penguji
> **"kenapa hyperplane pada SVM yang dipilih"**. Angka-angka di dalam kurung siku diisi dari
> keluaran cell di atas (tersimpan pula pada `hasil_svm_hyperplane.json`).

---

## Naskah jawaban

Pemilihan hyperplane pada model Support Vector Machine dalam penelitian ini tidak ditetapkan secara
sembarang, melainkan melalui tujuh eksperimen empiris. Secara teknis, memilih hyperplane berarti
menentukan tiga hal sekaligus: **kernel** (ruang tempat hyperplane dicari), **parameter C** (yang
mengatur trade-off antara lebar margin dan toleransi kesalahan pada formulasi soft margin), serta
**titik operasi** yang dianggap paling sesuai dengan tujuan sistem.

Pada tahap pertama, lima jenis kernel dibandingkan dalam pipeline yang identik
(StandardScaler - SMOTE - CalibratedClassifierCV), yaitu linear, RBF, polinomial derajat dua,
polinomial derajat tiga, dan sigmoid. Hasilnya, **kernel linear memberikan performa yang setara
dengan kernel non-linear** (selisih ROC-AUC hanya pada orde 0,00x), tetapi dengan waktu pelatihan
yang jauh lebih singkat. Pengujian lanjutan menggunakan *StratifiedKFold* lima lipatan yang
berpasangan, diikuti *paired t-test* dan uji *Wilcoxon signed-rank*, menunjukkan bahwa
**perbedaan antara kernel linear dan RBF tidak signifikan secara statistik pada taraf 5%**.
Berdasarkan prinsip *parsimony* (Occam's razor), ketika dua model tidak dapat dibedakan
performanya, dipilih model yang lebih sederhana - dalam hal ini kernel linear, yang selain lebih
cepat juga menghasilkan batas keputusan yang dapat dituliskan sebagai satu persamaan linear
sehingga dapat dijelaskan kepada tenaga medis.

Setelah ruang pencarian ditetapkan, parameter C disapu pada rentang 0,001 hingga 100. Untuk setiap
nilai C dihitung norma vektor bobot ||w||, lebar margin 2/||w||, jumlah *support vector*, serta
recall dan precision pada data uji. Pola yang muncul konsisten dengan teori: **semakin besar C,
margin semakin sempit**, sementara **recall tidak ikut membaik**. Dengan aturan pemilihan yang
ditetapkan di awal - yaitu memilih margin terlebar di antara konfigurasi yang recall-nya masih
berada dalam satu poin persen dari recall terbaik - diperoleh **C = 0,1**, yang sekaligus
mengonfirmasi hasil *hyperparameter tuning* pada tahap sebelumnya. Dengan demikian, hyperplane yang
dipilih adalah **hyperplane dengan margin terlebar yang tetap memenuhi kebutuhan sensitivitas
skrining medis**: margin lebar berarti model tidak terlalu menempel pada data latih dan lebih
mampu menggeneralisasi ke pasien baru, sedangkan syarat recall memastikan sistem tidak melewatkan
penderita diabetes (kesalahan *false negative* jauh lebih berbahaya daripada *false positive*
dalam konteks skrining).

Justifikasi tersebut diperkuat secara visual melalui penggambaran hyperplane pada bidang dua fitur
paling diskriminatif (HbA1c dan kadar glukosa darah) serta pada proyeksi dua komponen utama PCA
dari kelima fitur. Kedua visualisasi memperlihatkan garis keputusan beserta dua garis margin dan
posisi *support vector*, dan menunjukkan bahwa pemisahan antar-kelas memang mengikuti pola lurus.
Perbandingan visual berdampingan antara kernel linear dan RBF pada bidang yang sama menegaskan
bahwa kelengkungan yang dihasilkan RBF hanya mengikuti kerapatan lokal data di sekitar ambang
klinis dan tidak berubah menjadi keunggulan performa. Interpretasi vektor bobot w menunjukkan
kontribusi tiap fitur terhadap keputusan model, dengan arah pengaruh yang sesuai secara klinis, dan
urutan kepentingannya konsisten dengan hasil *permutation importance* yang bersifat model-agnostik.
Terakhir, karena sistem mengambil keputusan berdasarkan ambang probabilitas sementara SVM hanya
menghasilkan skor jarak ke hyperplane, keluaran model dikalibrasi menggunakan
`CalibratedClassifierCV`; perbandingan Brier score dan *reliability diagram* antara metode sigmoid,
isotonic, dan tanpa kalibrasi membuktikan bahwa kalibrasi tersebut memang diperlukan agar angka
risiko yang ditampilkan bermakna secara probabilistik.

Sebagai catatan keterbatasan yang disampaikan secara terbuka, eksperimen perbandingan kernel dan
grid parameter dijalankan pada **subsample stratified** (proporsi kelas dipertahankan), karena SVM
dengan kernel non-linear memiliki kompleksitas pelatihan sekitar O(n^2) hingga O(n^3) sehingga
tidak memungkinkan dilatih pada seluruh 96.146 baris data dengan sumber daya komputasi yang
tersedia. Kesimpulan yang ditarik karenanya bersifat **komparatif** - yaitu mengurutkan kernel dan
konfigurasi parameter secara relatif - dan bukan estimasi performa absolut. Model final dengan
kernel linear sendiri tetap dilatih pada seluruh data karena biaya komputasinya mendekati linear
terhadap jumlah sampel.

---

## Poin-poin singkat untuk sesi tanya jawab

1. **Kenapa linear?** Karena secara statistik tidak berbeda dari RBF (uji t berpasangan dan
   Wilcoxon, p >= 0,05), tetapi jauh lebih cepat, dapat dilatih pada seluruh data, dan dapat
   diinterpretasi.
2. **Kenapa C = 0,1?** Karena menghasilkan margin paling lebar (generalisasi terbaik) di antara
   konfigurasi yang recall-nya masih setara dengan yang terbaik - aturan ini ditetapkan sebelum
   melihat hasil, bukan sesudahnya.
3. **Apa bukti marginnya lebih lebar?** Tabel `tabel_analisis_margin` memuat ||w||, 2/||w||, jumlah
   support vector, dan rasio support vector untuk setiap nilai C, serta divisualisasikan pada
   `svm_margin_vs_C.png`.
4. **Kenapa RBF/poly/sigmoid ditolak?** Angka lengkapnya ada pada `tabel_perbandingan_kernel`:
   selisih AUC berada pada orde 0,00x sementara waktu latih berlipat-lipat.
5. **Kenapa perlu kalibrasi?** Karena sistem memakai ambang probabilitas; `tabel_kalibrasi_svm`
   dan `svm_kalibrasi.png` membuktikan perbaikan Brier score setelah kalibrasi.
6. **Kenapa memakai subsample?** Karena kompleksitas O(n^2)-O(n^3) pada kernel non-linear - ini
   dilaporkan sebagai keterbatasan, dan hanya memengaruhi klaim absolut, bukan perbandingan relatif.
